## Feature Engineering

## **Chat**

In [1]:
pip install nltk langdetect pymorphy2 setuptools==80

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 11.5 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 86.8 MB/s eta 0:00:00:00:0100:01
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=59ec55bbb05f7944e080853a0b70b7a499d6c8d177551cb85a3679cfda08b1e2
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
  Created wheel for docopt: filename=docopt-0.6.2-py2.py3-none-any.whl size=13706 sha256=4bacdc4e6b7e90e94fb34fbb14d3db070c5fba333f1281801cd6e597fae42f22
  Stored in directory: /root/.cache/pip/wheels/1a/bf/a1/4cee4f7678c68c5875ca89eaccf460593539805c3906722228
Successfully built langdetect docopt
  Attempting uninsta

In [3]:
import cuml
import pandas as pd
import langdetect
import pymorphy2
import re
import numpy as np
import kagglehub
import os
import nltk
import inspect
import sys
import joblib
from ast import literal_eval
from pymorphy2 import MorphAnalyzer
from cuml.linear_model import LogisticRegression as cmLogReg
from concurrent.futures import ProcessPoolExecutor
from collections import defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LogisticRegression as skLogReg
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder,MultiLabelBinarizer
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin
from scipy.sparse import hstack, csr_matrix
from nltk.tokenize import TweetTokenizer
from nltk.corpus import stopwords
from typing import Iterable, Optional

In [10]:
def gini(y_true, y_score):
    return 2 * roc_auc_score(y_true, y_score) - 1.0

In [4]:
#kaggle_dir = '/root/.kaggle'
#os.makedirs(kaggle_dir, exist_ok=True)

#!cp kaggle.json {kaggle_dir}/
#!chmod 600 {kaggle_dir}/kaggle.json


In [5]:
path_files = kagglehub.competition_download('dota-2-hse-ml-1-course-competition-2026')

In [6]:
chat = pd.read_csv(path_files + '/game_chat.csv')
chat.sample(5)

,match_id,radiant_chat,dire_chat
20865,660395,NaN,NaN
22800,438090,舒服,举报你
320500,17993,NaN,NaN
533350,271138,NaN,ну а зачем ты упоминаешь...
461752,578566,РАЗЪЕБА,NaN


In [7]:
chat.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 700838 entries, 0 to 700837
Data columns (total 3 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   match_id      700838 non-null  int64 
 1   radiant_chat  90624 non-null   object
 2   dire_chat     90604 non-null   object
dtypes: int64(1), object(2)
memory usage: 16.0+ MB


In [8]:
chat.isna().sum()

match_id             0
radiant_chat    610214
dire_chat       610234
dtype: int64

In [9]:
chat['dire_chat'].dropna().head(5)

0     саппортам | U LAGGING BRAH | stack..... | шлюх
6                                             vision
11                               juegan comoooo bots
19                                              经济碾压
26                                              排位真爽
Name: dire_chat, dtype: object

In [10]:
chat['dire_chat'].dropna().sample(5)

131652                   FKLDJSFLKJSLKF
645210                    СФ | мидеру))
587180    RUNES | yg nonton di watch 5k
64993                             ЛОББИ
283384                            CAGON
Name: dire_chat, dtype: object

It appears that the messages are separated by the `|` symbol.

In [11]:
nltk.download('stopwords')
language_map = {
    'ru': 'russian',
    'en': 'english',
    'de': 'german',
    'es': 'spanish',
    'fr': 'french',
    'it': 'italian',
    'pt': 'portuguese',
    'nl': 'dutch',
    'sv': 'swedish',
    'no': 'norwegian',
    'fi': 'finnish',
    'da': 'danish',
    'ar': 'arabic',
    'az': 'azerbaijani',
    'bn': 'bengali',
    'eu': 'basque',
    'ca': 'catalan',
    'zh': 'chinese',
    'hr': 'croatian',
    'cs': 'czech',
    'sk': 'slovak',
    'sl': 'slovenian',
    'et': 'estonian',
    'fa': 'persian',
    'el': 'greek',
    'he': 'hebrew',
    'hi': 'hindi',
    'hu': 'hungarian',
    'id': 'indonesian',
    'ga': 'irish',
    'ja': 'japanese',
    'kk': 'kazakh',
    'ko': 'korean',
    'ku': 'kurdish',
    'la': 'latin',
    'lv': 'latvian',
    'lt': 'lithuanian',
    'ms': 'malay',
    'mr': 'marathi',
    'ne': 'nepali',
    'pa': 'punjabi',
    'ro': 'romanian',
    'sd': 'sindhi',
    'so': 'somali',
    'sw': 'swahili',
    'tl': 'tagalog',
    'ta': 'tamil',
    'te': 'telugu',
    'th': 'thai',
    'tr': 'turkish',
    'uk': 'ukrainian',
    'ur': 'urdu',
    'uz': 'uzbek',
    'vi': 'vietnamese'
}

[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [12]:
def _getargspec_compat(func):
    fullspec = inspect.getfullargspec(func)
    return (fullspec.args, fullspec.varargs, fullspec.varkw, fullspec.defaults)

if not hasattr(inspect, 'getargspec') or sys.version_info >= (3, 10):
    inspect.getargspec = _getargspec_compat


In [13]:
def remove_repeated(text: str):
  return re.sub(r"(.)\1+", r"\1\1", text)

def preprocessing(text: str, tokenizer = TweetTokenizer(), morph = MorphAnalyzer()):
    text = text.split('|')
    preprocessed_message = []
    for message in text:
      prep = preprocessing_message(message, tokenizer, morph)
      preprocessed_message.extend(prep)
    return preprocessed_message

def preprocessing_message(text: str, tokenizer, morph):
    text = remove_repeated(text)
    try:
        detected_language_code = langdetect.detect(text)
        nltk_language_name = language_map.get(detected_language_code, 'english')
    except langdetect.lang_detect_exception.LangDetectException:
        nltk_language_name = 'english'

    try:
        stop_words = stopwords.words(nltk_language_name)
    except OSError:
        stop_words = stopwords.words('english')

    tokenized = tokenizer.tokenize(text)
    tokenized = [tok for tok in tokenized if tok.lower() not in stop_words]

    lemmatized = [morph.parse(tok)[0].normal_form if tok.isalpha() else tok for tok in tokenized]

    return lemmatized

print(preprocessing("Ляяя, ваша мама такая красивая, ну вылитый пудж)))0"))

['ляя', ',', 'ваш', 'мама', 'такой', 'красивый', ',', 'вылитый', 'пуджа', ')', ')', '0']


In [14]:
def parallel_preprocessing(texts):
    with ProcessPoolExecutor() as executor:
        return list(executor.map(preprocessing, texts))

In [15]:
dire_chat = chat[['match_id', 'dire_chat']].dropna()
dire_chat.head()

,match_id,dire_chat
0,235435,саппортам | U LAGGING BRAH | stack..... | шлюх
6,390519,vision
11,192321,juegan comoooo bots
19,422105,经济碾压
26,639411,排位真爽


In [16]:
radiant_chat = chat[['match_id', 'radiant_chat']].dropna()
radiant_chat.head()

,match_id,radiant_chat
0,235435,потренируйся с ботами????
2,383046,u just buy levels blue? | fa ge
13,653306,no ulti | красавчики
18,749401,one sec
19,422105,reaaaady


In [17]:
def process_in_batches_with_parallel(df, team ,batch_size=1000):
    result = []
    for i in range(0, len(df), batch_size):
        batch = df[i:i+batch_size]
        try:
            result.extend(parallel_preprocessing(batch[f'{team}_chat']))
        except Exception as e:
            print(f"Error processing batch_{i} - {i+batch_size}: {e}")
            continue
    return result

In [18]:
dire_chat['dire_chat_tokenized'] = process_in_batches_with_parallel(dire_chat, 'dire', batch_size=10000)
dire_chat.head()

,match_id,dire_chat,dire_chat_tokenized
0,235435,саппортам | U LAGGING BRAH | stack..... | шлюх,"[саппорт, u, lagging, brah, stack, .., шлюха]"
6,390519,vision,[vision]
11,192321,juegan comoooo bots,"[juegan, comoo, bots]"
19,422105,经济碾压,[经济碾压]
26,639411,排位真爽,[排位真爽]


In [19]:
radiant_chat['radiant_chat_tokenized'] = process_in_batches_with_parallel(radiant_chat, 'radiant', batch_size=10000)
radiant_chat.head()

,match_id,radiant_chat,radiant_chat_tokenized
0,235435,потренируйся с ботами????,"[потренироваться, бот, ?, ?]"
2,383046,u just buy levels blue? | fa ge,"[u, just, buy, levels, blue, ?, fa, ge]"
13,653306,no ulti | красавчики,"[no, ulti, красавчик]"
18,749401,one sec,"[one, sec]"
19,422105,reaaaady,[reaady]


In [20]:
chat = chat.merge(radiant_chat[['match_id', 'radiant_chat_tokenized']], on='match_id', how = 'left')
chat.head()

,match_id,radiant_chat,dire_chat,radiant_chat_tokenized
0,235435,потренируйся с ботами????,саппортам | U LAGGING BRAH | stack..... | шлюх,"[потренироваться, бот, ?, ?]"
1,102127,NaN,NaN,NaN
2,383046,u just buy levels blue? | fa ge,NaN,"[u, just, buy, levels, blue, ?, fa, ge]"
3,729879,NaN,NaN,NaN
4,126886,NaN,NaN,NaN


In [21]:
chat = chat.merge(dire_chat[['match_id', 'dire_chat_tokenized']], on='match_id', how = 'left')
chat.head()

,match_id,radiant_chat,dire_chat,radiant_chat_tokenized,dire_chat_tokenized
0,235435,потренируйся с ботами????,саппортам | U LAGGING BRAH | stack..... | шлюх,"[потренироваться, бот, ?, ?]","[саппорт, u, lagging, brah, stack, .., шлюха]"
1,102127,NaN,NaN,NaN,NaN
2,383046,u just buy levels blue? | fa ge,NaN,"[u, just, buy, levels, blue, ?, fa, ge]",NaN
3,729879,NaN,NaN,NaN,NaN
4,126886,NaN,NaN,NaN,NaN


In [22]:
chat.to_csv('chat_tokenized.csv')

### **Vectorization** 


In [23]:
#chat = pd.read_csv('/kaggle/input/datasets/qwerte123/computed-data-dota-hse/chat_tokenized.csv')
dire_chat = chat[['match_id', 'dire_chat', 'dire_chat_tokenized']].dropna()
radiant_chat = chat[['match_id', 'radiant_chat', 'radiant_chat_tokenized']].dropna()
dire_chat.head()

,match_id,dire_chat,dire_chat_tokenized
0,235435,саппортам | U LAGGING BRAH | stack..... | шлюх,"[саппорт, u, lagging, brah, stack, .., шлюха]"
6,390519,vision,[vision]
11,192321,juegan comoooo bots,"[juegan, comoo, bots]"
19,422105,经济碾压,[经济碾压]
26,639411,排位真爽,[排位真爽]


#### I perform vectorization separately during cross-validation to avoid data leakage. I do not want information from the chat logs of future games to find its way into the training data.

In [24]:
def vectorize(texts, vectorizer):
    texts_joined = []
    for text in texts:
        cur_text = " ".join(text)
        texts_joined.append(cur_text)
    res = vectorizer.fit_transform(texts_joined)
    return res, vectorizer


tfidf_dire = TfidfVectorizer()
tfidf_radiant = TfidfVectorizer()

dire_tfidf_matrix, tfidf_dire = vectorize(dire_chat['dire_chat_tokenized'], tfidf_dire)
radiant_tfidf_matrix, tfidf_radiant = vectorize(radiant_chat['radiant_chat_tokenized'], tfidf_radiant)

pd.DataFrame(dire_tfidf_matrix.toarray(), columns=tfidf_dire.get_feature_names_out())

,00,01,02,041,05,07,08,09,0_0,0adsijku9ddfajmnik,...,ｐｌｅａｓｅ,ｓｅｘ,ｓｈｉｔ,ｓｍａｃｋｅｄ,ｓｔｒｏｎｇｅｒ,ｔｈａｎ,ｔｈｅｒｅ,ｔｈｉｓ,ｔｏ,ｗｅｅｄ
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
90599,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
90600,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
90601,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
90602,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [25]:
train = pd.read_csv(path_files + '/matches_df_train.csv')
test = pd.read_csv(path_files + '/matches_df_test.csv')
players = pd.read_csv(path_files + '/player_df.csv')
heroes = pd.read_csv(path_files + '/Constants.Heroes.csv')

In [26]:
class HeroesEncoder:
    def fit(self, heroes_df, y=None):
      self.unique_heroes = heroes_df['id']
      self.heroes_to_id = {hero: id for id, hero in enumerate(self.unique_heroes)}
      return self

    def transform(self, X, y=None):
      unique_matches = X['match_id'].unique()
      matches_to_id = {match: id for id, match in enumerate(unique_matches)}
      slots = X['player_slot'].values
      data = np.zeros((len(unique_matches), len(self.unique_heroes)), dtype = int)
      rows = X['match_id'].map(matches_to_id)
      cols = X['hero_id'].map(self.heroes_to_id)
      radiant = (slots<=4) & (slots>=0)
      dire = (slots >= 128) & (slots <= 132)
      data[rows[radiant], cols[radiant]] = 1
      data[rows[dire], cols[dire]] = -1
      data = np.column_stack([unique_matches,data])
      return data


In [27]:
def players_merge(df, players, heroes, df_cols, mode = 'train'):
    players = players.copy()
    players_grouped = players.groupby(['match_id','hero_id'])['hero_id'].count()
    wrong_games = players_grouped[players_grouped != 1].index.get_level_values(0).unique()
    players = players[~players['match_id'].isin(wrong_games)]

    matches_hero0 = players[players['hero_id']==0]['match_id'].unique()
    players = players[~players['match_id'].isin(matches_hero0)]

    radiant, dire = range(5), range(128, 133)
    players['team'] = ''
    players.loc[players['player_slot'].isin(radiant), 'team'] = 'radiant'
    players.loc[players['player_slot'].isin(dire), 'team'] = 'dire'
    grouped_matches = players.groupby(['match_id', 'account_id'])['team'].nunique()
    double_players = grouped_matches[(grouped_matches>1) & (~grouped_matches.index.get_level_values('account_id').isin([-1, 4294967295]))].reset_index()
    players = players[~players['match_id'].isin(double_players['match_id'].unique())]

    matches = df['match_id'].unique()
    players = players[players['match_id'].isin(matches)]

    hero_encoder = HeroesEncoder()
    hero_encoder.fit(heroes)
    match_heropool = hero_encoder.transform(players)

    match_heropool_df = pd.DataFrame(match_heropool, columns = ['match_id'] + [f'hero_{i+1}' for i in range(match_heropool.shape[1]-1)])
    if mode == 'train':
      heropool_features = pd.merge(df[df_cols],match_heropool_df, how='inner', on='match_id')
    elif mode == 'test':
      heropool_features = pd.merge(df[df_cols],match_heropool_df, how='left', on='match_id')
      heropool_features.fillna(0, inplace=True)

    return heropool_features


In [28]:
train_heroes = players_merge(train, players, heroes,train.columns, mode='train')
test_heroes = players_merge(test, players, heroes, test.columns, mode='test')

passthrough_columns = ['match_id', 'duration']
num_features = ['avg_mmr']
region_features = ['region']
game_mode_features = ['game_mode']
date_features = ['date']

#### The `MissingIndicator` class adds an indicator column for the absence of the match's average MMR. The `DateAdder` class adds the remaining preprocessing steps performed in the first part (one-hot encoding of regions, date-related features, etc.).

In [29]:
class MissingIndicator(BaseEstimator, TransformerMixin):
  def __init__(self):
    self.columns_names = None

  def fit(self, X, y=None):
    self.columns_names = X.columns.tolist()
    return self

  def transform(self, X):
    X_copy = X.copy()
    for col in self.columns_names:
      X_copy[f'{col}_missing'] = X_copy[col].isna()
    return X_copy

  def get_feature_names_out(self, input_features=None):
    if input_features is None:
      input_features = self.columns_names
    return input_features + ([f'{col}_missing' for col in input_features])


class DateAdder(BaseEstimator, TransformerMixin):
  def __init__(self, date_column='date'):
    self.columns_names = None
    self.date_column = date_column

  def fit(self, X, y=None):
    return self

  def transform(self, X):
    X_copy = X.copy()
    X_copy[self.date_column] = pd.to_datetime(X_copy[self.date_column])
    X_copy['day'] = X_copy[self.date_column].dt.day
    X_copy['day_week'] = X_copy[self.date_column].dt.dayofweek
    X_copy['is_weekend'] = X_copy[self.date_column].dt.weekday >=5
    seasons = {'spring': [3,4,5], 'summer': [6,7,8], 'fall':[9,10,11], 'winter':[12,1,2]}
    reversed_seasons = {l: k  for k, v in seasons.items() for l in v}
    X_copy['season'] = X_copy[self.date_column].dt.month.map(reversed_seasons)
    X_copy = X_copy.drop(self.date_column, axis=1)
    self.columns_names = X_copy.columns.tolist()
    return X_copy

  def get_feature_names_out(self, input_features=None):
    return self.columns_names

In [30]:
imputer = SimpleImputer(strategy='median')
sqrt_transformer = FunctionTransformer(func=np.sqrt, feature_names_out='one-to-one')
pipe_num = Pipeline([('missing_ind', MissingIndicator()),('imputer', imputer),
                     ('transformer', sqrt_transformer), ('scaler', RobustScaler())])

ohe = OneHotEncoder(sparse_output=False)
pipe_reg = Pipeline([('ecoder', ohe)])

pipe_game_mode = Pipeline([('encoder', ohe)])

pipe_date = Pipeline([('adder', DateAdder()), ('ecoder', ColumnTransformer([
     ('weekend', 'passthrough', ['is_weekend']),
     ('ohe', ohe, ['day', 'day_week', 'season'])
], verbose_feature_names_out=False))])

preprocessor = ColumnTransformer([
    ('passthrough', 'passthrough', passthrough_columns),
    ('num', pipe_num, num_features),
    ('reg', pipe_reg, region_features),
    ('game_mode', pipe_game_mode, game_mode_features),
    ('date', pipe_date, date_features)
], verbose_feature_names_out=False)

train_prep = preprocessor.fit_transform(train)
columns_names = passthrough_columns+ preprocessor['num'].get_feature_names_out().tolist() \
 + preprocessor['reg'].get_feature_names_out().tolist() \
 + preprocessor['game_mode'].get_feature_names_out().tolist() \
 + preprocessor['date'].get_feature_names_out().tolist()
train_prep = pd.DataFrame(train_prep, columns=columns_names)
train_prep.head()

,match_id,duration,avg_mmr,avg_mmr_missing,region_Australia,region_China,region_Europe East,region_Europe West,region_Middle East,region_SE Asia,...,day_week_1,day_week_2,day_week_3,day_week_4,day_week_5,day_week_6,season_fall,season_spring,season_summer,season_winter
0,1.0,2625.0,0.371672,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,2.0,7526.0,-0.315982,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2,3.0,2831.0,-0.847251,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
3,4.0,1438.0,0.000000,1.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
4,5.0,2051.0,0.000000,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0


In [31]:
train_prep = train_prep.merge(train[['match_id', 'radiant_win']], how='left', on='match_id')
train_prep_heroes = players_merge(train_prep, players, heroes, train_prep.columns)
train_prep_heroes.head()

,match_id,duration,avg_mmr,avg_mmr_missing,region_Australia,region_China,region_Europe East,region_Europe West,region_Middle East,region_SE Asia,...,hero_117,hero_118,hero_119,hero_120,hero_121,hero_122,hero_123,hero_124,hero_125,hero_126
0,1.0,2625.0,0.371672,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
1,2.0,7526.0,-0.315982,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
2,3.0,2831.0,-0.847251,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0,0,0,0,0,0,0,0,0,0
3,4.0,1438.0,0.000000,1.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0,0,0,0,0,0,0,0,0,0
4,5.0,2051.0,0.000000,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0


In [32]:
train_chat = train_prep_heroes.merge(
    chat[['match_id', 'dire_chat_tokenized', 'radiant_chat_tokenized']],
    on='match_id', how='left')

train_chat['dire_chat_tokenized'] = train_chat['dire_chat_tokenized'].apply(
    lambda x: x if isinstance(x, list) else [])
train_chat['radiant_chat_tokenized'] = train_chat['radiant_chat_tokenized'].apply(
    lambda x: x if isinstance(x, list) else [])

train_chat.head()

,match_id,duration,avg_mmr,avg_mmr_missing,region_Australia,region_China,region_Europe East,region_Europe West,region_Middle East,region_SE Asia,...,hero_119,hero_120,hero_121,hero_122,hero_123,hero_124,hero_125,hero_126,dire_chat_tokenized,radiant_chat_tokenized
0,1.0,2625.0,0.371672,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0,0,0,0,0,0,0,0,[],"[225, ?, ?]"
1,2.0,7526.0,-0.315982,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,[],[]
2,3.0,2831.0,-0.847251,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0,0,0,0,0,0,0,0,[],[]
3,4.0,1438.0,0.000000,1.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0,0,0,0,0,0,0,0,[],[]
4,5.0,2051.0,0.000000,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,[],[]


In [33]:
def prepare_texts(series):
    result = []
    for x in series:
        if isinstance(x, list):
            result.append(" ".join(map(str, x)))
        elif x is None or (isinstance(x, float) and np.isnan(x)):
            result.append("")
        else:
            result.append(str(x))
    return result

In [34]:
def print_score(scores):
  for i, s in enumerate(scores):
    print(f'Fold {i+1}: {s}')
  print(f'Average value: {np.mean(scores)}')

def train_cv(df, params, features, model = cmLogReg):
      cv = TimeSeriesSplit(n_splits=5)
      scores = []
      y_train_df = df['radiant_win']
      X_train_df = df[features]
      for fold, (train_index, val_index) in enumerate(cv.split(df)):
          X_train, X_val = X_train_df.iloc[train_index], X_train_df.iloc[val_index]
          y_train, y_val = y_train_df.iloc[train_index], y_train_df.iloc[val_index]
          cur_model = model(**params)
          cur_model.fit(X_train, y_train)
          predict_val = cur_model.decision_function(X_val)
          score = gini(y_val, predict_val)
          scores.append(score)
      print_score(scores)
      
      cur_model = model(**params)
      cur_model.fit(X_train_df, y_train_df)
      return scores, cur_model


scores_base, model_base = train_cv(train_prep_heroes, {}, 
                                   train_prep_heroes.columns[1:].drop(['radiant_win', 'duration']),
                                  model = cmLogReg)

Fold 1: 0.3225509565969631
Fold 2: 0.2703735668952414
Fold 3: 0.30112732591171065
Fold 4: 0.30060211765909006
Fold 5: 0.3093504844007655
Average value: 0.30080089029275414


In [35]:
def train_cv_vectorize(df, dire_texts, radiant_texts,
                       features,params = {}, model = cmLogReg):
    cv = TimeSeriesSplit(n_splits=5)
    scores = []
    X_all_tab = csr_matrix(df[features].to_numpy())
    
    for fold, (train_index, val_index) in enumerate(cv.split(df)):
        X_train_tab = X_all_tab[train_index]
        X_val_tab = X_all_tab[val_index]
        y_train = df['radiant_win'][train_index]
        y_val = df['radiant_win'][val_index]
    
        dire_train = [dire_texts[i] for i in train_index]
        dire_val = [dire_texts[i] for i in val_index]
        rad_train = [radiant_texts[i] for i in train_index]
        rad_val = [radiant_texts[i] for i in val_index]
    
        tfidf_dire = TfidfVectorizer(min_df=3, max_features=30000)
        tfidf_radiant = TfidfVectorizer(min_df=3, max_features=30000)
        X_train_dire = tfidf_dire.fit_transform(dire_train)
        X_val_dire = tfidf_dire.transform(dire_val)
        X_train_radiant = tfidf_radiant.fit_transform(rad_train)
        X_val_radiant = tfidf_radiant.transform(rad_val)
        X_train = hstack([X_train_tab, X_train_dire, X_train_radiant], format='csr')
        X_val = hstack([X_val_tab, X_val_dire, X_val_radiant], format='csr')
    
        cur_model = model(**params)
        cur_model.fit(X_train, y_train)
    
        pred_val = cur_model.decision_function(X_val)
        score = gini(y_val, pred_val)
        scores.append(score)
        
    print_score(scores)
    tfidf_dire_final = TfidfVectorizer(min_df=3, max_features=30000)
    tfidf_radiant_final = TfidfVectorizer(min_df=3, max_features=30000)
    
    X_all_dire = tfidf_dire_final.fit_transform(dire_texts)
    X_all_radiant = tfidf_radiant_final.fit_transform(radiant_texts)
    X_all = hstack([X_all_tab, X_all_dire, X_all_radiant], format='csr')   
    
    final_model = model(**params)
    final_model.fit(X_all, df['radiant_win'])
    
    return tfidf_dire_final, tfidf_radiant_final, scores, final_model

In [36]:
target_col = 'radiant_win'
drop_cols = ['match_id', 'radiant_win', 'duration', 'dire_chat_tokenized', 'radiant_chat_tokenized']

feature_cols = [c for c in train_chat.columns if c not in drop_cols]

dire_texts = prepare_texts(train_chat['dire_chat_tokenized'])
radiant_texts = prepare_texts(train_chat['radiant_chat_tokenized'])
tfidf_dire_final, tfidf_radiant_final,scores_vect, model_vect = train_cv_vectorize(train_chat, dire_texts, radiant_texts,
                                             feature_cols)

Fold 1: 0.34350431802405357
Fold 2: 0.29781141227389774
Fold 3: 0.3300341234454942
Fold 4: 0.33175509720365337
Fold 5: 0.3418442080603774
Average value: 0.32898983180149527


We observe a clear improvement in cross-validation relative to the baseline when using vectorization.

In [37]:
test['duration'] = train['duration'].median()
test_prep = preprocessor.transform(test)
test_columns = columns_names
test_prep = pd.DataFrame(test_prep, columns=test_columns)
test_prep_heroes = players_merge(test_prep, players, heroes, test_prep.columns, mode='test')

In [38]:
test_full = test_prep_heroes.merge(
    chat[['match_id', 'dire_chat_tokenized', 'radiant_chat_tokenized']],
    on='match_id', how='left')

test_full['dire_chat_tokenized'] = test_full['dire_chat_tokenized'].apply(
    lambda x: x if isinstance(x, list) else [])

test_full['radiant_chat_tokenized'] = test_full['radiant_chat_tokenized'].apply(
    lambda x: x if isinstance(x, list) else [])

test_feature_cols = [c for c in test_full.columns if c not in ['match_id', 'duration', 'dire_chat_tokenized', 'radiant_chat_tokenized']]
X_test_tab = csr_matrix(test_full[test_feature_cols].astype(float).to_numpy())

dire_test_texts = prepare_texts(test_full['dire_chat_tokenized'])
radiant_test_texts = prepare_texts(test_full['radiant_chat_tokenized'])

X_test_dire = tfidf_dire_final.transform(dire_test_texts)
X_test_radiant = tfidf_radiant_final.transform(radiant_test_texts)
X_test = hstack([X_test_tab, X_test_dire, X_test_radiant], format='csr')

test_pred = model_vect.predict_proba(X_test)[:, 1]

In [39]:
submit_vec = pd.DataFrame({"ID":test_full['match_id'].astype(int), 
                           "Value": test_pred})
submit_vec.head()

,ID,Value
0,8,0.410234
1,29,0.645919
2,34,0.742727
3,36,0.479700
4,61,0.598309


In [40]:
submit_vec.to_csv('submit_vec.csv', index = False)

## **Aggregations**

In [41]:
advances = pd.read_csv(path_files + '/dota_adv.csv')
advances.head()

,match_id,radiant_gold_adv,radiant_exp_adv
0,526846,[ 0 159 452 1904 2100 3290 3290 3290 3290 ...,[ 0 68 658 1397 1435 2118 2118 1923 1923 ...
1,511496,[ 0 -151 -141 12 -165 -151 -151 4 377 ...,[ 0 1 -136 243 -270 -8 -8 -169 -169 ...
2,90272,[],[]
3,153647,[],[]
4,694826,[],[]


In [42]:
advances.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 767822 entries, 0 to 767821
Data columns (total 3 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   match_id          767822 non-null  int64 
 1   radiant_gold_adv  767822 non-null  object
 2   radiant_exp_adv   767822 non-null  object
dtypes: int64(1), object(2)
memory usage: 17.6+ MB


In [43]:
advances.isnull().sum()

match_id            0
radiant_gold_adv    0
radiant_exp_adv     0
dtype: int64

In [44]:
gold_val_counts = advances['radiant_gold_adv'].value_counts()
num_gold_nan = len(advances[advances['radiant_gold_adv'] == '[]'])
gold_val_counts.head(7)

radiant_gold_adv
[]                                                                                                     529783
[   0  167  257 1303 1503 1400 1400 1577 1353 1353 1830 1830 1830 3379\n 3379 6392]                         1
[   0  169  250  420 1206 1427 1427 1138 1411 2050 2050 1337 -224  393\n  393 2473]                         1
[   0  126 1092  579  535  796  796  796  571  796 1270 2029 1873 2908\n 2908 2167]                         1
[   0 -326 -214 -537 -787 -219 -219 -219  123 -102 -102  621  272 -843\n -843 -317]                         1
[    0    -4  -411  -275  -585 -1088 -1088 -1298 -1026 -1026  -460  -948\n -1899 -1899 -1057 -1057]         1
[    0  -149 -1012 -1135 -1741 -3221 -3221 -3352 -3615 -3615 -3498 -3149\n -3721 -4715 -4715 -4715]         1
Name: count, dtype: int64

In [45]:
exp_val_counts = advances['radiant_exp_adv'].value_counts()
num_exp_nan = len(advances[advances['radiant_exp_adv'] == '[]'])
exp_val_counts.head(7)

radiant_exp_adv
[]                                                                                                     529783
[    0   145   256   662   562   446   446   291   -86   133   689   227\n  -772  -196 -1424  -618]         1
[   0    1 -332 -678 -220  199  199  395  395 1095  764 1786 1786 3335\n 2719 1624]                         1
[    0  -205   672   302   100   365   365   365   -47  -790 -1828 -1693\n -1081 -2041 -2041  -668]         1
[   0  -27   64  135 -206 -385 -385 -513 -513 -110  711 2047 2047 3585\n 2975 2975]                         1
[    0    83  -464  -445  -803 -1251 -1251 -1251 -1251 -1001 -1001 -1612\n -1010 -1010 -1807  -793]         1
[    0  -305  -833  -946 -1226 -2792 -2792 -2607 -2147 -1422 -1422 -1422\n -1422  -350   677   677]         1
Name: count, dtype: int64

In [46]:
print(f"The actual rate of information missings, predominantly in gold: {num_gold_nan/len(advances)}")
print(f"The actual rate of information missings, predominantly in exp: {num_exp_nan/len(advances)}")

The actual rate of information missings, predominantly in gold: 0.6899815321780308
The actual rate of information missings, predominantly in exp: 0.6899815321780308


In [47]:
advances['radiant_gold_adv'] = advances['radiant_gold_adv'].apply(lambda x: x.replace('[', '').replace(']', '').split())
advances['radiant_gold_adv'] = advances['radiant_gold_adv'].apply(lambda x: list(map(int, x)))

advances['radiant_exp_adv'] = advances['radiant_exp_adv'].apply(lambda x: x.replace('[', '').replace(']', '').split())
advances['radiant_exp_adv'] = advances['radiant_exp_adv'].apply(lambda x: list(map(int, x)))

advances.head()

,match_id,radiant_gold_adv,radiant_exp_adv
0,526846,"[0, 159, 452, 1904, 2100, 3290, 3290, 3290, 32...","[0, 68, 658, 1397, 1435, 2118, 2118, 1923, 192..."
1,511496,"[0, -151, -141, 12, -165, -151, -151, 4, 377, ...","[0, 1, -136, 243, -270, -8, -8, -169, -169, 18..."
2,90272,[],[]
3,153647,[],[]
4,694826,[],[]


In [48]:
advances['num_observ_gold'] = advances['radiant_gold_adv'].apply(len)
advances['num_observ_exp'] = advances['radiant_exp_adv'].apply(len)
advances.head()

,match_id,radiant_gold_adv,radiant_exp_adv,num_observ_gold,num_observ_exp
0,526846,"[0, 159, 452, 1904, 2100, 3290, 3290, 3290, 32...","[0, 68, 658, 1397, 1435, 2118, 2118, 1923, 192...",16,16
1,511496,"[0, -151, -141, 12, -165, -151, -151, 4, 377, ...","[0, 1, -136, 243, -270, -8, -8, -169, -169, 18...",16,16
2,90272,[],[],0,0
3,153647,[],[],0,0
4,694826,[],[],0,0


In [49]:
advances['num_observ_gold'].value_counts()

num_observ_gold
0     529783
16    238039
Name: count, dtype: int64

In [50]:
advances['num_observ_exp'].value_counts()

num_observ_exp
0     529783
16    238039
Name: count, dtype: int64

In [51]:
advances[advances['num_observ_exp'] != advances['num_observ_gold']]

,match_id,radiant_gold_adv,radiant_exp_adv,num_observ_gold,num_observ_exp


We have nearly 69% missing values in our data - which is a very serious issue. On one hand, we are reluctant to simply discard such a large volume of data; on the other, it is unclear how to impute it, given that we have no other information available in the table -and building a model to predict the sequence would be extremely difficult and unlikely to be effective.

In [52]:
advances_nonan = advances.copy()
advances_nonan = advances_nonan.drop(['num_observ_exp', 'num_observ_gold'], axis=1)
advances_nonan.head()

,match_id,radiant_gold_adv,radiant_exp_adv
0,526846,"[0, 159, 452, 1904, 2100, 3290, 3290, 3290, 32...","[0, 68, 658, 1397, 1435, 2118, 2118, 1923, 192..."
1,511496,"[0, -151, -141, 12, -165, -151, -151, 4, 377, ...","[0, 1, -136, 243, -270, -8, -8, -169, -169, 18..."
2,90272,[],[]
3,153647,[],[]
4,694826,[],[]


In [53]:
def base_agg(df, stat='gold'):
    df = df.copy()
    col_stat = [col for col in df.columns if stat in col]
    for col in col_stat:
        df[f'{col}_avg'] = df[col].apply(lambda x: np.mean(x) if len(x) > 0 else np.nan)
        df[f'{col}_std'] = df[col].apply(lambda x: np.std(x) if len(x) > 1 else np.nan)
        df[f'{col}_lst'] = df[col].apply(lambda x: x[-1] if len(x) > 0 else np.nan)
        df[f'{col}_kurtosis'] = df[col].apply(lambda x: pd.Series(x).kurt() if len(x) > 3 else np.nan)

    return df

advances_based = base_agg(advances_nonan)
advances_based = base_agg(advances_based, 'exp')
advances_based.head()

,match_id,radiant_gold_adv,radiant_exp_adv,radiant_gold_adv_avg,radiant_gold_adv_std,radiant_gold_adv_lst,radiant_gold_adv_kurtosis,radiant_exp_adv_avg,radiant_exp_adv_std,radiant_exp_adv_lst,radiant_exp_adv_kurtosis
0,526846,"[0, 159, 452, 1904, 2100, 3290, 3290, 3290, 32...","[0, 68, 658, 1397, 1435, 2118, 2118, 1923, 192...",2887.500,1559.052797,5342.0,-0.199244,2262.250,1401.628406,3311.0,-0.659761
1,511496,"[0, -151, -141, 12, -165, -151, -151, 4, 377, ...","[0, 1, -136, 243, -270, -8, -8, -169, -169, 18...",933.875,1342.763702,3698.0,-0.144060,191.625,385.377717,201.0,0.234210
2,90272,[],[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,153647,[],[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,694826,[],[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


I will break down the final value by team - specifically, I will modify it to ensure that the advantage cannot drop below zero - thereby eliminating the linear dependency between the features.

In [54]:
advances_based['dire_exp_adv_lst'] = advances_based['radiant_exp_adv_lst'].apply(lambda x: -x if x < 0 else 0)
advances_based['dire_gold_adv_lst'] = advances_based['radiant_gold_adv_lst'].apply(lambda x: -x if x < 0 else 0)

advances_based['radiant_exp_adv_lst'] = advances_based['radiant_exp_adv_lst'].apply(lambda x: x if x > 0 else 0)
advances_based['radiant_gold_adv_lst'] = advances_based['radiant_gold_adv_lst'].apply(lambda x: x if x > 0 else 0)

advances_based.head()

,match_id,radiant_gold_adv,radiant_exp_adv,radiant_gold_adv_avg,radiant_gold_adv_std,radiant_gold_adv_lst,radiant_gold_adv_kurtosis,radiant_exp_adv_avg,radiant_exp_adv_std,radiant_exp_adv_lst,radiant_exp_adv_kurtosis,dire_exp_adv_lst,dire_gold_adv_lst
0,526846,"[0, 159, 452, 1904, 2100, 3290, 3290, 3290, 32...","[0, 68, 658, 1397, 1435, 2118, 2118, 1923, 192...",2887.500,1559.052797,5342.0,-0.199244,2262.250,1401.628406,3311.0,-0.659761,0.0,0.0
1,511496,"[0, -151, -141, 12, -165, -151, -151, 4, 377, ...","[0, 1, -136, 243, -270, -8, -8, -169, -169, 18...",933.875,1342.763702,3698.0,-0.144060,191.625,385.377717,201.0,0.234210,0.0,0.0
2,90272,[],[],NaN,NaN,0.0,NaN,NaN,NaN,0.0,NaN,0.0,0.0
3,153647,[],[],NaN,NaN,0.0,NaN,NaN,NaN,0.0,NaN,0.0,0.0
4,694826,[],[],NaN,NaN,0.0,NaN,NaN,NaN,0.0,NaN,0.0,0.0


In [55]:
advances_based['is_nan'] = advances_based['radiant_exp_adv_kurtosis'].isnull()
advances_based.head()

,match_id,radiant_gold_adv,radiant_exp_adv,radiant_gold_adv_avg,radiant_gold_adv_std,radiant_gold_adv_lst,radiant_gold_adv_kurtosis,radiant_exp_adv_avg,radiant_exp_adv_std,radiant_exp_adv_lst,radiant_exp_adv_kurtosis,dire_exp_adv_lst,dire_gold_adv_lst,is_nan
0,526846,"[0, 159, 452, 1904, 2100, 3290, 3290, 3290, 32...","[0, 68, 658, 1397, 1435, 2118, 2118, 1923, 192...",2887.500,1559.052797,5342.0,-0.199244,2262.250,1401.628406,3311.0,-0.659761,0.0,0.0,False
1,511496,"[0, -151, -141, 12, -165, -151, -151, 4, 377, ...","[0, 1, -136, 243, -270, -8, -8, -169, -169, 18...",933.875,1342.763702,3698.0,-0.144060,191.625,385.377717,201.0,0.234210,0.0,0.0,False
2,90272,[],[],NaN,NaN,0.0,NaN,NaN,NaN,0.0,NaN,0.0,0.0,True
3,153647,[],[],NaN,NaN,0.0,NaN,NaN,NaN,0.0,NaN,0.0,0.0,True
4,694826,[],[],NaN,NaN,0.0,NaN,NaN,NaN,0.0,NaN,0.0,0.0,True


#### Hyperparameter tuning results obtained on the baseline DataFrame

In [58]:
base_params = joblib.load('/kaggle/input/datasets/qwerte123/computed-data-dota-hse/logreg_heropool_model.pkl')
print(base_params)

{'solver': 'lbfgs', 'C': 242.17452442894617, 'max_iter': 1997}


In [59]:
train_heroes = players_merge(train_prep, players, heroes,train_prep.columns, mode='train')
test_heroes = players_merge(test_prep, players, heroes, test_prep.columns, mode='test')

In [60]:
train_heroes.head()

,match_id,duration,avg_mmr,avg_mmr_missing,region_Australia,region_China,region_Europe East,region_Europe West,region_Middle East,region_SE Asia,...,hero_117,hero_118,hero_119,hero_120,hero_121,hero_122,hero_123,hero_124,hero_125,hero_126
0,1.0,2625.0,0.371672,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
1,2.0,7526.0,-0.315982,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
2,3.0,2831.0,-0.847251,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0,0,0,0,0,0,0,0,0,0
3,4.0,1438.0,0.000000,1.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0,0,0,0,0,0,0,0,0,0
4,5.0,2051.0,0.000000,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0


In [61]:
base_scores, base_model = train_cv(train_heroes, base_params, 
                                   train_heroes.columns[1:].drop(['duration', 'radiant_win']),
                                   skLogReg)

Fold 1: 0.32254521055716756
Fold 2: 0.27034060668369353
Fold 3: 0.3011094122730258
Fold 4: 0.30064959866267804
Fold 5: 0.30936832608587106
Average value: 0.3008026308524872


In [62]:
advances_train_heroes = pd.merge(train_heroes, advances_based, how='left', on='match_id')
advances_train_heroes = advances_train_heroes.drop(['radiant_gold_adv', 'radiant_exp_adv'], axis=1)

nulls = advances_train_heroes.isnull().sum()[advances_train_heroes.isnull().sum()>0]
nulls_cols = nulls.index.tolist()
nulls.head()

radiant_gold_adv_avg         441845
radiant_gold_adv_std         441845
radiant_gold_adv_kurtosis    441845
radiant_exp_adv_avg          441845
radiant_exp_adv_std          441845
dtype: int64

In [63]:
for col in nulls_cols:
  advances_train_heroes[col] = advances_train_heroes[col].fillna(0)

In [64]:
advances_train_heroes

,match_id,duration,avg_mmr,avg_mmr_missing,region_Australia,region_China,region_Europe East,region_Europe West,region_Middle East,region_SE Asia,...,radiant_gold_adv_std,radiant_gold_adv_lst,radiant_gold_adv_kurtosis,radiant_exp_adv_avg,radiant_exp_adv_std,radiant_exp_adv_lst,radiant_exp_adv_kurtosis,dire_exp_adv_lst,dire_gold_adv_lst,is_nan
0,1.0,2625.0,0.371672,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,True
1,2.0,7526.0,-0.315982,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,True
2,3.0,2831.0,-0.847251,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,True
3,4.0,1438.0,0.000000,1.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,True
4,5.0,2051.0,0.000000,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
640025,767815.0,1888.0,-2.074504,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,True
640026,767817.0,2789.0,0.000000,1.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,True
640027,767818.0,1197.0,-0.570071,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,True
640028,767820.0,4155.0,0.000000,1.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,True


In [65]:
features_advances = advances_train_heroes.columns[2:].drop('radiant_win')

In [66]:
advances_train_heroes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 640030 entries, 0 to 640029
Columns: 209 entries, match_id to is_nan
dtypes: bool(2), float64(81), int64(126)
memory usage: 1012.0 MB


In [67]:
scores_advances, model_advances = train_cv(advances_train_heroes, {}, features_advances)

[2026-04-07 13:42:36.035] [CUML] [warning] L-BFGS line search failed (code 4); stopping at the last valid step
[2026-04-07 13:42:51.017] [CUML] [warning] L-BFGS line search failed (code 4); stopping at the last valid step
Fold 1: 0.3225345403954982
Fold 2: 0.2703321705855921
Fold 3: 0.32607209274925175
Fold 4: -0.08560038238968948
Fold 5: 0.21259028459010176
Average value: 0.20918574118615085
[2026-04-07 13:43:08.101] [CUML] [warning] L-BFGS line search failed (code 4); stopping at the last valid step


#### As is evident, scaling is simply essential here

In [68]:
scale_cols_based = advances_based.columns.drop(['match_id', 'is_nan', 'radiant_gold_adv', 'radiant_exp_adv'])
scaler = StandardScaler()

scaled = scaler.fit_transform(advances_based[scale_cols_based])
scaled = pd.DataFrame(scaled, columns = scaler.get_feature_names_out())

advances_based[scale_cols_based] = scaled
advances_based.head()

,match_id,radiant_gold_adv,radiant_exp_adv,radiant_gold_adv_avg,radiant_gold_adv_std,radiant_gold_adv_lst,radiant_gold_adv_kurtosis,radiant_exp_adv_avg,radiant_exp_adv_std,radiant_exp_adv_lst,radiant_exp_adv_kurtosis,dire_exp_adv_lst,dire_gold_adv_lst,is_nan
0,526846,"[0, 159, 452, 1904, 2100, 3290, 3290, 3290, 32...","[0, 68, 658, 1397, 1435, 2118, 2118, 1923, 192...",0.003604,-0.004053,3.107569,-0.548738,1.904949,0.710432,1.954684,-0.747951,-0.278383,-0.276066,False
1,511496,"[0, -151, -141, 12, -165, -151, -151, 4, 377, ...","[0, 1, -136, 243, -270, -8, -8, -169, -169, 18...",0.003014,-0.004108,2.041192,-0.523619,0.017353,-1.084811,-0.220841,-0.337852,-0.278383,-0.276066,False
2,90272,[],[],NaN,NaN,-0.357508,NaN,NaN,NaN,-0.361446,NaN,-0.278383,-0.276066,True
3,153647,[],[],NaN,NaN,-0.357508,NaN,NaN,NaN,-0.361446,NaN,-0.278383,-0.276066,True
4,694826,[],[],NaN,NaN,-0.357508,NaN,NaN,NaN,-0.361446,NaN,-0.278383,-0.276066,True


In [69]:
advances_train_heroes_scaled = pd.merge(train_heroes,
                                        advances_based, 
                                        how='left', on='match_id')
advances_train_heroes_scaled = advances_train_heroes_scaled.drop(['radiant_gold_adv', 'radiant_exp_adv'], axis=1)

nulls = advances_train_heroes_scaled.isnull().sum()[advances_train_heroes_scaled.isnull().sum()>0]
nulls_cols = nulls.index.tolist()
nulls.head()

radiant_gold_adv_avg         441845
radiant_gold_adv_std         441845
radiant_gold_adv_kurtosis    441845
radiant_exp_adv_avg          441845
radiant_exp_adv_std          441845
dtype: int64

In [70]:
for col in nulls_cols:
  advances_train_heroes_scaled[col] = advances_train_heroes_scaled[col].fillna(0)

In [71]:
scores_advances, model_advances = train_cv(advances_train_heroes_scaled, {}, features_advances)

Fold 1: 0.32256986281934563
Fold 2: 0.2704067553280345
Fold 3: 0.3395976608286344
Fold 4: 0.5313147800708098
Fold 5: 0.3065623380274347
Average value: 0.3540902794148518


In [72]:
advances_test_heroes = pd.merge(test_heroes, advances_based,
                                how='left', on='match_id')
advances_test_heroes = advances_test_heroes.drop(['radiant_gold_adv', 'radiant_exp_adv'], axis=1)

nulls = advances_test_heroes.isnull().sum()[advances_test_heroes.isnull().sum()>0]
nulls_cols = nulls.index.tolist()
nulls

radiant_gold_adv_avg         41103
radiant_gold_adv_std         41103
radiant_gold_adv_kurtosis    41103
radiant_exp_adv_avg          41103
radiant_exp_adv_std          41103
radiant_exp_adv_kurtosis     41103
dtype: int64

In [73]:
for col in nulls_cols:
  advances_test_heroes[col] = advances_test_heroes[col].fillna(0)

In [74]:
submit = model_advances.decision_function(advances_test_heroes[features_advances])
submit_df = pd.DataFrame({"ID":advances_test_heroes['match_id'].astype(int).tolist(), "Value": submit})
submit_df.head()

,ID,Value
0,8,-0.300928
1,29,0.548539
2,34,0.517604
3,36,0.110709
4,61,0.298210


In [75]:
submit_df.to_csv('submit_prep.csv', index=False)

Our results improved during cross-validation - which is already a good sign-meaning it is time to move forward in the same direction.

In [76]:
def more_agg(df, stat='gold'):
    df = df.copy()

    col_stat = [
        col for col in df.columns
        if stat in col and not any(x in col for x in ['avg', 'std', 'lst', 'kurtosis'])
    ]

    for col in col_stat:
        df[f'{col}_min'] = df[col].apply(lambda x: np.min(x) if len(x) > 0 else np.nan)
        df[f'{col}_max'] = df[col].apply(lambda x: np.max(x) if len(x) > 0 else np.nan)
        df[f'{col}_med'] = df[col].apply(lambda x: np.median(x) if len(x) > 0 else np.nan)
        df[f'{col}_p25'] = df[col].apply(lambda x: np.percentile(x, 0.25) if len(x) > 0 else np.nan)
        df[f'{col}_p75'] = df[col].apply(lambda x: np.percentile(x, 0.75) if len(x) > 0 else np.nan)

    return df

advances_based_more = more_agg(advances_based, 'gold')
advances_based_more = more_agg(advances_based_more, 'exp')
advances_based_more.head()

,match_id,radiant_gold_adv,radiant_exp_adv,radiant_gold_adv_avg,radiant_gold_adv_std,radiant_gold_adv_lst,radiant_gold_adv_kurtosis,radiant_exp_adv_avg,radiant_exp_adv_std,radiant_exp_adv_lst,...,radiant_gold_adv_min,radiant_gold_adv_max,radiant_gold_adv_med,radiant_gold_adv_p25,radiant_gold_adv_p75,radiant_exp_adv_min,radiant_exp_adv_max,radiant_exp_adv_med,radiant_exp_adv_p25,radiant_exp_adv_p75
0,526846,"[0, 159, 452, 1904, 2100, 3290, 3290, 3290, 32...","[0, 68, 658, 1397, 1435, 2118, 2118, 1923, 192...",0.003604,-0.004053,3.107569,-0.548738,1.904949,0.710432,1.954684,...,0.0,5342.0,3290.0,5.9625,17.8875,0.0,4661.0,2020.5,2.5500,7.6500
1,511496,"[0, -151, -141, 12, -165, -151, -151, 4, 377, ...","[0, 1, -136, 243, -270, -8, -8, -169, -169, 18...",0.003014,-0.004108,2.041192,-0.523619,0.017353,-1.084811,-0.220841,...,-165.0,3698.0,156.0,-164.4750,-163.4250,-270.0,931.0,93.5,-266.2125,-258.6375
2,90272,[],[],NaN,NaN,-0.357508,NaN,NaN,NaN,-0.361446,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,153647,[],[],NaN,NaN,-0.357508,NaN,NaN,NaN,-0.361446,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,694826,[],[],NaN,NaN,-0.357508,NaN,NaN,NaN,-0.361446,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [77]:
advances_train_heroes_more = pd.merge(train_heroes, 
                                      advances_based_more, 
                                      how='left', on='match_id')
advances_train_heroes_more = advances_train_heroes_more.drop(['radiant_gold_adv', 
                                                              'radiant_exp_adv'], axis=1)

nulls = advances_train_heroes_more.isnull().sum()[advances_train_heroes_more.isnull().sum()>0]
nulls_cols = nulls.index.tolist()

for col in nulls_cols:
    advances_train_heroes_more[col] = advances_train_heroes_more[col].fillna(0)

features_advances_more =advances_train_heroes_more.columns[2:].drop('radiant_win')
scores_advances_more, model_advances_more = train_cv(advances_train_heroes_more, 
                                                     {}, features_advances_more)

[2026-04-07 13:46:21.835] [CUML] [warning] L-BFGS line search failed (code 4); stopping at the last valid step
[2026-04-07 13:46:37.638] [CUML] [warning] L-BFGS line search failed (code 4); stopping at the last valid step
Fold 1: 0.3225549596221162
Fold 2: 0.27038353478339405
Fold 3: 0.18404700340727342
Fold 4: -0.2534695458333407
Fold 5: 0.21262874446702673
Average value: 0.14722893928929395
[2026-04-07 13:46:55.426] [CUML] [warning] L-BFGS line search failed (code 4); stopping at the last valid step


#### The previous step wasn't strictly necessary, I just added some additional features, but now comes the scaling.

In [78]:
scale_cols_based_more = advances_based_more.columns.drop(['match_id', 'is_nan', 
                                                          'radiant_gold_adv', 'radiant_exp_adv'])

scaler = StandardScaler()
scaled = scaler.fit_transform(advances_based_more[scale_cols_based_more])
scaled = pd.DataFrame(scaled, columns = scaler.get_feature_names_out())
advances_based_more[scale_cols_based_more] = scaled

advances_train_heroes_more = pd.merge(train_heroes, 
                                      advances_based_more, 
                                      how='left', on='match_id')
advances_train_heroes_more = advances_train_heroes_more.drop(['radiant_gold_adv', 'radiant_exp_adv'], axis=1)

nulls = advances_train_heroes_more.isnull().sum()[advances_train_heroes_more.isnull().sum()>0]
nulls_cols = nulls.index.tolist()

for col in nulls_cols:
    advances_train_heroes_more[col] = advances_train_heroes_more[col].fillna(0)
features_advances_more =advances_train_heroes_more.columns[2:].drop('radiant_win')
scores_advances_more, model_advances_more = train_cv(advances_train_heroes_more, {}, features_advances_more)

[2026-04-07 13:47:48.893] [CUML] [warning] L-BFGS line search failed (code 3); stopping at the last valid step
Fold 1: 0.32257222833921717
Fold 2: 0.2703146341213898
Fold 3: 0.3415121114192736
Fold 4: 0.5342230245984623
Fold 5: 0.3062670992958243
Average value: 0.35497781955483343


#### We observe a slight improvement compared to the variant with basic aggregation

In [79]:
advances_test_heroes_more = pd.merge(test_heroes, advances_based_more,
                                how='left', on='match_id')
advances_test_heroes_more = advances_test_heroes_more.drop(['radiant_gold_adv', 
                                                            'radiant_exp_adv'], axis=1)

nulls = advances_test_heroes_more.isnull().sum()[advances_test_heroes_more.isnull().sum()>0]
nulls_cols = nulls.index.tolist()
for col in nulls_cols:
  advances_test_heroes_more[col] = advances_test_heroes_more[col].fillna(0)

submit = model_advances_more.decision_function(advances_test_heroes_more[features_advances_more])
submit_df = pd.DataFrame({"ID":advances_test_heroes_more['match_id'].astype(int).tolist(), "Value": submit})
submit_df.to_csv('submit_more_agg.csv', index=False)

### Trends 

#### Here we add trends of game features

In [80]:
import gc
gc.collect()

0

In [81]:
class TrendTransformer:

    def __init__(self, columns: Iterable[str], method = 'delta', join_col = 'match_id'):
        self.columns = columns
        self.method = method
        self.join_col = join_col
        self.slope = None
        self.interception = None
        self.r2 = None

    def fit(self, X, y=None):
      return self

    def transform(self, values):
        self.r2 = {col: [] for col in self.columns}
        self.slope = {col: [] for col in self.columns}
        self.interception = {col: [] for col in self.columns}
        for col in self.columns:
            for cur_values in values[col]:
                cur_val = np.asarray(cur_values)
                X = np.arange(len(cur_values))
                if len(cur_val) == 0:
                  cur_slope = np.nan
                  cur_interception = np.nan
                  cur_r2 = np.nan

                if len(cur_val) == 1:
                  cur_slope = np.nan
                  cur_interception = cur_val[0]
                  cur_r2 = np.nan

                if self.method == 'delta':
                  if len(cur_val) > 1:
                    cur_slope = (cur_val[-1] - cur_val[0])/(len(X) - 1) # y = kx + b
                    cur_interception = cur_val[0]
                    y_mean = np.mean(cur_val)
                    y_pred = cur_slope * X + cur_interception
                    RSS = np.sum((y_pred - cur_val)**2)
                    TSS = np.sum((cur_val - y_mean)**2)
                    cur_r2 = 1 - RSS/TSS if TSS != 0 else np.nan

                elif self.method == 'OLS':
                  if len(cur_val) > 1:
                    X_ols = np.column_stack([X, np.full(fill_value = 1, shape=(cur_val.shape))])
                    X_ols_t = X_ols.T
                    beta = np.linalg.inv(X_ols_t @ X_ols) @ X_ols_t @ cur_val
                    cur_slope = beta[0]
                    cur_interception = beta[1]
                    y_mean = np.mean(cur_val)
                    y_pred = cur_slope * X + cur_interception
                    RSS = np.sum((y_pred - cur_val)**2)
                    TSS = np.sum((cur_val - y_mean)**2)
                    cur_r2 = 1 - RSS/TSS if TSS != 0 else np.nan
                self.r2[col].append(cur_r2)
                self.slope[col].append(cur_slope)
                self.interception[col].append(cur_interception)

        res = pd.DataFrame(values[self.join_col])
        for col in self.columns:
          res[f'{col}_r2'] = self.r2[col]
          res[f'{col}_slope'] = self.slope[col]
          res[f'{col}_interception'] = self.interception[col]
        return res


In [82]:
tt = TrendTransformer(['radiant_gold_adv', 'radiant_exp_adv'], method='OLS')
trends = tt.transform(advances_based_more)
trends.head()

,match_id,radiant_gold_adv_r2,radiant_gold_adv_slope,radiant_gold_adv_interception,radiant_exp_adv_r2,radiant_exp_adv_slope,radiant_exp_adv_interception
0,526846,0.810015,304.388235,604.588235,0.863183,282.491176,143.566176
1,511496,0.773514,256.185294,-987.514706,0.263800,42.938235,-130.411765
2,90272,NaN,NaN,NaN,NaN,NaN,NaN
3,153647,NaN,NaN,NaN,NaN,NaN,NaN
4,694826,NaN,NaN,NaN,NaN,NaN,NaN


In [83]:
scaler = StandardScaler()

to_be_scaled = trends.columns.drop('match_id')
trends_scaled = scaler.fit_transform(trends[to_be_scaled])
trends_scaled = pd.DataFrame(trends_scaled, columns=scaler.get_feature_names_out())
trends[to_be_scaled] = trends_scaled

In [84]:
advances_trends = pd.merge(advances_based_more, trends, how='left', on='match_id')
advances_trends.head()

,match_id,radiant_gold_adv,radiant_exp_adv,radiant_gold_adv_avg,radiant_gold_adv_std,radiant_gold_adv_lst,radiant_gold_adv_kurtosis,radiant_exp_adv_avg,radiant_exp_adv_std,radiant_exp_adv_lst,...,radiant_exp_adv_max,radiant_exp_adv_med,radiant_exp_adv_p25,radiant_exp_adv_p75,radiant_gold_adv_r2,radiant_gold_adv_slope,radiant_gold_adv_interception,radiant_exp_adv_r2,radiant_exp_adv_slope,radiant_exp_adv_interception
0,526846,"[0, 159, 452, 1904, 2100, 3290, 3290, 3290, 32...","[0, 68, 658, 1397, 1435, 2118, 2118, 1923, 192...",0.003604,-0.004053,3.107569,-0.548738,1.904949,0.710432,1.954684,...,1.244551,2.106467,0.858268,0.857712,1.217054,-0.002601,0.002992,1.446850,1.255258,0.380877
1,511496,"[0, -151, -141, 12, -165, -151, -151, 4, 377, ...","[0, 1, -136, 243, -270, -8, -8, -169, -169, 18...",0.003014,-0.004108,2.041192,-0.523619,0.017353,-1.084811,-0.220841,...,-0.545857,0.028710,0.688921,0.687713,1.089439,-0.002669,0.002807,-0.665578,-0.029601,0.068825
2,90272,[],[],NaN,NaN,-0.357508,NaN,NaN,NaN,-0.361446,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,153647,[],[],NaN,NaN,-0.357508,NaN,NaN,NaN,-0.361446,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,694826,[],[],NaN,NaN,-0.357508,NaN,NaN,NaN,-0.361446,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [85]:
advances_trends_train_heroes = pd.merge(train_heroes, advances_trends, how='left', on='match_id')
advances_trends_train_heroes = advances_trends_train_heroes.drop(['radiant_gold_adv', 'radiant_exp_adv'], axis=1)

nulls_trends = advances_trends_train_heroes.isnull().sum()[advances_trends_train_heroes.isnull().sum()>0]
nulls_cols = nulls_trends.index.tolist()
nulls_trends

radiant_gold_adv_avg             441845
radiant_gold_adv_std             441845
radiant_gold_adv_kurtosis        441845
radiant_exp_adv_avg              441845
radiant_exp_adv_std              441845
radiant_exp_adv_kurtosis         441845
radiant_gold_adv_min             441845
radiant_gold_adv_max             441845
radiant_gold_adv_med             441845
radiant_gold_adv_p25             441845
radiant_gold_adv_p75             441845
radiant_exp_adv_min              441845
radiant_exp_adv_max              441845
radiant_exp_adv_med              441845
radiant_exp_adv_p25              441845
radiant_exp_adv_p75              441845
radiant_gold_adv_r2              441845
radiant_gold_adv_slope           441845
radiant_gold_adv_interception    441845
radiant_exp_adv_r2               441845
radiant_exp_adv_slope            441845
radiant_exp_adv_interception     441845
dtype: int64

In [86]:
for col in nulls_cols:
  advances_trends_train_heroes[col] = advances_trends_train_heroes[col].fillna(0)

features_trends = advances_trends_train_heroes.columns[2:].drop('radiant_win').tolist()

features_trends.remove('radiant_exp_adv_interception')
features_trends.remove('radiant_gold_adv_interception')

advances_trends_train_heroes.columns
cols = ['radiant_gold_adv_r2', 'radiant_gold_adv_slope',
       'radiant_gold_adv_interception', 'radiant_exp_adv_r2',
       'radiant_exp_adv_slope', 'radiant_exp_adv_interception']

In [87]:
scores_trends, model_trends = train_cv(advances_trends_train_heroes, {}, features_trends)

[2026-04-07 13:49:34.164] [CUML] [warning] L-BFGS line search failed (code 3); stopping at the last valid step
Fold 1: 0.3225793681576703
Fold 2: 0.2702869212402239
Fold 3: 0.34081704479520614
Fold 4: 0.5364512025365988
Fold 5: 0.30606749474017114
Average value: 0.3552404062939741


In [88]:
advances_trends_test_heroes = pd.merge(test_heroes, advances_trends, how='left', on='match_id')
advances_trends_test_heroes = advances_trends_test_heroes.drop(['radiant_gold_adv', 'radiant_exp_adv'], axis=1)

nulls_trends = advances_trends_test_heroes.isnull().sum()[advances_trends_test_heroes.isnull().sum()>0]
nulls_cols = nulls_trends.index.tolist()
nulls_trends

radiant_gold_adv_avg             41103
radiant_gold_adv_std             41103
radiant_gold_adv_kurtosis        41103
radiant_exp_adv_avg              41103
radiant_exp_adv_std              41103
radiant_exp_adv_kurtosis         41103
radiant_gold_adv_min             41103
radiant_gold_adv_max             41103
radiant_gold_adv_med             41103
radiant_gold_adv_p25             41103
radiant_gold_adv_p75             41103
radiant_exp_adv_min              41103
radiant_exp_adv_max              41103
radiant_exp_adv_med              41103
radiant_exp_adv_p25              41103
radiant_exp_adv_p75              41103
radiant_gold_adv_r2              41103
radiant_gold_adv_slope           41103
radiant_gold_adv_interception    41103
radiant_exp_adv_r2               41103
radiant_exp_adv_slope            41103
radiant_exp_adv_interception     41103
dtype: int64

In [89]:
for col in nulls_cols:
  fill_value = 0
  advances_trends_test_heroes[col] = advances_trends_test_heroes[col].fillna(fill_value)

submit = model_trends.decision_function(advances_trends_test_heroes[features_trends])
submit_df = pd.DataFrame({"ID":advances_trends_test_heroes['match_id'].astype(int).tolist(), "Value": submit})
submit_df.head()

,ID,Value
0,8,-0.295857
1,29,0.542306
2,34,0.521555
3,36,0.108085
4,61,0.362699


In [90]:
submit_df.to_csv('submit_trends.csv', index=False)

#### Binarization 

We create bins of advantages 

In [91]:
class BinTransformer:
  def __init__(self, columns: Iterable[str], join_col = 'match_id', number_bins=4):
    self.columns = columns
    self.join_col = join_col
    self.number_bins = number_bins

  def fit(self, X, y=None):
    return self

  def transform(self, values):
    res = defaultdict(list)
    for col in self.columns:
        for cur_values in values[col]:
          cur_values = np.asarray(cur_values)
          if len(cur_values) == 0:
              for i in range(self.number_bins):
                res[f'{col}_bin_{i}'].append([np.nan])
              continue
          step = len(cur_values)//self.number_bins
          for i in range(self.number_bins):
                bin_i = cur_values[i*step:(i+1)*step]
                res[f'{col}_bin_{i}'].append(bin_i)
    res = pd.DataFrame(res)
    res[self.join_col] = values[self.join_col]
    return res

In [92]:
Bt = BinTransformer(['radiant_gold_adv', 'radiant_exp_adv'], number_bins=4)
bins_cols = Bt.transform(advances_trends)
bins_cols.head()

,radiant_gold_adv_bin_0,radiant_gold_adv_bin_1,radiant_gold_adv_bin_2,radiant_gold_adv_bin_3,radiant_exp_adv_bin_0,radiant_exp_adv_bin_1,radiant_exp_adv_bin_2,radiant_exp_adv_bin_3,match_id
0,"[0, 159, 452, 1904]","[2100, 3290, 3290, 3290]","[3290, 3859, 3859, 3087]","[3087, 3849, 5342, 5342]","[0, 68, 658, 1397]","[1435, 2118, 2118, 1923]","[1923, 1923, 2494, 3772]","[3734, 4661, 4661, 3311]",526846
1,"[0, -151, -141, 12]","[-165, -151, -151, 4]","[377, 300, 1222, 2130]","[2130, 2130, 3698, 3698]","[0, 1, -136, 243]","[-270, -8, -8, -169]","[-169, 186, 931, 931]","[931, 201, 201, 201]",511496
2,[nan],[nan],[nan],[nan],[nan],[nan],[nan],[nan],90272
3,[nan],[nan],[nan],[nan],[nan],[nan],[nan],[nan],153647
4,[nan],[nan],[nan],[nan],[nan],[nan],[nan],[nan],694826


In [93]:
bins_based = base_agg(bins_cols)
bins_based = base_agg(bins_based,'exp')
bins_based = more_agg(bins_based)
bins_based = more_agg(bins_based, 'exp')
bins_based.head()

,radiant_gold_adv_bin_0,radiant_gold_adv_bin_1,radiant_gold_adv_bin_2,radiant_gold_adv_bin_3,radiant_exp_adv_bin_0,radiant_exp_adv_bin_1,radiant_exp_adv_bin_2,radiant_exp_adv_bin_3,match_id,radiant_gold_adv_bin_0_avg,...,radiant_exp_adv_bin_2_min,radiant_exp_adv_bin_2_max,radiant_exp_adv_bin_2_med,radiant_exp_adv_bin_2_p25,radiant_exp_adv_bin_2_p75,radiant_exp_adv_bin_3_min,radiant_exp_adv_bin_3_max,radiant_exp_adv_bin_3_med,radiant_exp_adv_bin_3_p25,radiant_exp_adv_bin_3_p75
0,"[0, 159, 452, 1904]","[2100, 3290, 3290, 3290]","[3290, 3859, 3859, 3087]","[3087, 3849, 5342, 5342]","[0, 68, 658, 1397]","[1435, 2118, 2118, 1923]","[1923, 1923, 2494, 3772]","[3734, 4661, 4661, 3311]",526846,628.75,...,1923.0,3772.0,2208.5,1923.0000,1923.0000,3311.0,4661.0,4197.5,3314.1725,3320.5175
1,"[0, -151, -141, 12]","[-165, -151, -151, 4]","[377, 300, 1222, 2130]","[2130, 2130, 3698, 3698]","[0, 1, -136, 243]","[-270, -8, -8, -169]","[-169, 186, 931, 931]","[931, 201, 201, 201]",511496,-70.00,...,-169.0,931.0,558.5,-166.3375,-161.0125,201.0,931.0,201.0,201.0000,201.0000
2,[nan],[nan],[nan],[nan],[nan],[nan],[nan],[nan],90272,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,[nan],[nan],[nan],[nan],[nan],[nan],[nan],[nan],153647,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,[nan],[nan],[nan],[nan],[nan],[nan],[nan],[nan],694826,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [94]:
scaling_cols = bins_based.columns.drop([col for col in bins_based.columns if 'bin' in col[-6:] or col == 'match_id'])
bins_based_scaled = scaler.fit_transform(bins_based[scaling_cols])
bins_based[scaling_cols] = bins_based_scaled
bins_based.head()

,radiant_gold_adv_bin_0,radiant_gold_adv_bin_1,radiant_gold_adv_bin_2,radiant_gold_adv_bin_3,radiant_exp_adv_bin_0,radiant_exp_adv_bin_1,radiant_exp_adv_bin_2,radiant_exp_adv_bin_3,match_id,radiant_gold_adv_bin_0_avg,...,radiant_exp_adv_bin_2_min,radiant_exp_adv_bin_2_max,radiant_exp_adv_bin_2_med,radiant_exp_adv_bin_2_p25,radiant_exp_adv_bin_2_p75,radiant_exp_adv_bin_3_min,radiant_exp_adv_bin_3_max,radiant_exp_adv_bin_3_med,radiant_exp_adv_bin_3_p25,radiant_exp_adv_bin_3_p75
0,"[0, 159, 452, 1904]","[2100, 3290, 3290, 3290]","[3290, 3859, 3859, 3087]","[3087, 3849, 5342, 5342]","[0, 68, 658, 1397]","[1435, 2118, 2118, 1923]","[1923, 1923, 2494, 3772]","[3734, 4661, 4661, 3311]",526846,0.002985,...,1.783490,2.314638,1.672791,1.782428,1.780282,1.543568,1.201045,1.497338,1.543288,1.542708
1,"[0, -151, -141, 12]","[-165, -151, -151, 4]","[377, 300, 1222, 2130]","[2130, 2130, 3698, 3698]","[0, 1, -136, 243]","[-270, -8, -8, -169]","[-169, 186, 931, 931]","[931, 201, 201, 201]",511496,0.002896,...,0.134358,0.206231,0.339701,0.134654,0.135246,0.251053,-0.244200,-0.152749,0.249113,0.245226
2,[nan],[nan],[nan],[nan],[nan],[nan],[nan],[nan],90272,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,[nan],[nan],[nan],[nan],[nan],[nan],[nan],[nan],153647,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,[nan],[nan],[nan],[nan],[nan],[nan],[nan],[nan],694826,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [95]:
advances_bins = advances_based.merge(bins_based, how='left', on='match_id')
advances_bins.head()

,match_id,radiant_gold_adv,radiant_exp_adv,radiant_gold_adv_avg,radiant_gold_adv_std,radiant_gold_adv_lst,radiant_gold_adv_kurtosis,radiant_exp_adv_avg,radiant_exp_adv_std,radiant_exp_adv_lst,...,radiant_exp_adv_bin_2_min,radiant_exp_adv_bin_2_max,radiant_exp_adv_bin_2_med,radiant_exp_adv_bin_2_p25,radiant_exp_adv_bin_2_p75,radiant_exp_adv_bin_3_min,radiant_exp_adv_bin_3_max,radiant_exp_adv_bin_3_med,radiant_exp_adv_bin_3_p25,radiant_exp_adv_bin_3_p75
0,526846,"[0, 159, 452, 1904, 2100, 3290, 3290, 3290, 32...","[0, 68, 658, 1397, 1435, 2118, 2118, 1923, 192...",0.003604,-0.004053,3.107569,-0.548738,1.904949,0.710432,1.954684,...,1.783490,2.314638,1.672791,1.782428,1.780282,1.543568,1.201045,1.497338,1.543288,1.542708
1,511496,"[0, -151, -141, 12, -165, -151, -151, 4, 377, ...","[0, 1, -136, 243, -270, -8, -8, -169, -169, 18...",0.003014,-0.004108,2.041192,-0.523619,0.017353,-1.084811,-0.220841,...,0.134358,0.206231,0.339701,0.134654,0.135246,0.251053,-0.244200,-0.152749,0.249113,0.245226
2,90272,[],[],NaN,NaN,-0.357508,NaN,NaN,NaN,-0.361446,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,153647,[],[],NaN,NaN,-0.357508,NaN,NaN,NaN,-0.361446,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,694826,[],[],NaN,NaN,-0.357508,NaN,NaN,NaN,-0.361446,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [96]:
advances_bins_train_heroes = pd.merge(train_heroes, advances_bins, how='left', on='match_id')
advances_bins_train_heroes = advances_bins_train_heroes.drop(['radiant_gold_adv', 'radiant_exp_adv'], axis=1)

for col in advances_bins_train_heroes.columns:
    if 'bin' in col[-6:]:
        advances_bins_train_heroes = advances_bins_train_heroes.drop(col, axis=1)

nulls_trends = advances_bins_train_heroes.isnull().sum()[advances_bins_train_heroes.isnull().sum()>0]
nulls_cols = nulls_trends.index.tolist()
nulls_trends

for col in nulls_cols:
  advances_bins_train_heroes[col] = advances_bins_train_heroes[col].fillna(0)

features_bins = advances_bins_train_heroes.columns[2:].drop('radiant_win').tolist()
scores_bins, model_bins = train_cv(advances_bins_train_heroes, {'max_iter': 5000}, features_bins)

Fold 1: 0.3225842370629788
Fold 2: 0.2703126251414625
Fold 3: 0.41936152862975096
Fold 4: 0.5374201171731374
Fold 5: 0.30602456213008167
Average value: 0.3711406140274823


In [97]:
gc.collect()

0

#### Binarization yields a significant improvement in cross-validation

In [98]:
advances_bins_test_heroes = pd.merge(test_heroes, advances_bins, how='left', on='match_id')
advances_bins_test_heroes = advances_bins_test_heroes.drop(['radiant_gold_adv', 'radiant_exp_adv'], axis=1)

for col in advances_bins_test_heroes.columns:
    if 'bin' in col[-6:]:
        advances_bins_test_heroes = advances_bins_test_heroes.drop(col, axis=1)

In [99]:
for col in nulls_cols:
  advances_bins_test_heroes[col] = advances_bins_test_heroes[col].fillna(0)

submit = model_bins.decision_function(advances_bins_test_heroes[features_bins])
submit_df = pd.DataFrame({"ID":advances_bins_test_heroes['match_id'].astype(int).tolist(), "Value": submit})
submit_df.head()

,ID,Value
0,8,-0.298146
1,29,0.539816
2,34,0.521439
3,36,0.104486
4,61,0.496272


In [100]:
submit_df.to_csv('submit_bins.csv', index=False)

In [101]:
advances_all = pd.merge(advances_trends, bins_based, on = 'match_id', how = 'left')

In [102]:
gc.collect()

0

In [103]:
advances_all_train_heroes = pd.merge(train_heroes, advances_all, how='left', on='match_id')
advances_all_train_heroes = advances_all_train_heroes.drop(['radiant_gold_adv', 'radiant_exp_adv'], axis=1)

for col in advances_all_train_heroes.columns:
    if 'bin' in col[-6:]:
        advances_all_train_heroes = advances_all_train_heroes.drop(col, axis=1)

nulls_all = advances_all_train_heroes.isnull().sum()[advances_all_train_heroes.isnull().sum()>0]
nulls_cols = nulls_all.index.tolist()

for col in nulls_cols:
  advances_all_train_heroes[col] = advances_all_train_heroes[col].fillna(0)
features_all = advances_all_train_heroes.columns[2:].drop('radiant_win').tolist()

In [ ]:
scores_all, model_all = train_cv(advances_all_train_heroes, {'max_iter': 10000},
                                 features_all )

#### Combining all features yields the best result

In [ ]:
advances_all_test_heroes = pd.merge(test_heroes, advances_all, how='left', on='match_id')
advances_all_test_heroes = advances_all_test_heroes.drop(['radiant_gold_adv', 'radiant_exp_adv'], axis=1)

for col in advances_all_test_heroes.columns:
    if 'bin' in col[-6:]:
        advances_all_test_heroes = advances_all_test_heroes.drop(col, axis=1)

for col in nulls_cols:
    advances_all_test_heroes[col] = advances_all_test_heroes[col].fillna(0)

submit = model_all.decision_function(advances_all_test_heroes[features_all])
submit_df = pd.DataFrame({"ID":advances_all_test_heroes['match_id'].tolist(), "Value": submit})
submit_df['ID'] = submit_df['ID'].astype(int)
submit_df.to_csv('submit_all.csv',index=False)

In [105]:
import gc

chat = chat[['match_id', 'dire_chat_tokenized', 'radiant_chat_tokenized']].copy()

for var in [
    'players', 'heroes',
    'train_prep', 'train_prep_heroes',
    
    'dire_tfidf_matrix', 'radiant_tfidf_matrix',
    'advances_bins',
    'X_tab', 'y', 'dire_texts', 'radiant_texts',
    'scores'
]:
    if var in globals():
        del globals()[var]

gc.collect()

88

#### Let`s combine aggregations and chat vectorization

In [ ]:
chat = pd.read_csv('/kaggle/input/datasets/qwerte123/computed-data-dota-hse/chat_tokenized.csv')
chat.head()

In [ ]:
chat = chat.drop('Unnamed: 0', axis=1)
chat.head()

In [ ]:
def prepare_texts(series):
    result = []
    for x in series:
        if isinstance(x, list):
            result.append(" ".join(map(str, x)))
        elif x is None or (isinstance(x, float) and np.isnan(x)):
            result.append("")
        else:
            result.append(str(x))
    return result

In [ ]:
chat

In [ ]:
train_chat = train_prep_heroes.merge(
    chat[['match_id', 'dire_chat_tokenized', 'radiant_chat_tokenized']],
    on='match_id', how='left')

train_chat['dire_chat_tokenized'] = prepare_texts(train_chat['dire_chat_tokenized'])
train_chat['radiant_chat_tokenized'] = prepare_texts(train_chat['radiant_chat_tokenized'])

train_chat.head()

In [ ]:
chat_cols = ['match_id', 'dire_chat_tokenized', 'radiant_chat_tokenized']
train_full = advances_all_train_heroes.merge(train_chat[chat_cols], on='match_id',
    how='left')

drop_cols = ['match_id', 'radiant_win', 'duration', 'dire_chat_tokenized', 'radiant_chat_tokenized']

train_full['dire_chat_tokenized'] = train_full['dire_chat_tokenized']

train_full['radiant_chat_tokenized'] = train_full['radiant_chat_tokenized']

feature_cols = [c for c in train_full.columns if c not in drop_cols]

dire_train_texts = train_full['dire_chat_tokenized'].tolist()
radiant_train_texts = train_full['radiant_chat_tokenized'].tolist()

train_full.head()

In [ ]:
train_full['is_nan'] = train_full['is_nan'].astype(int)

In [ ]:
tfidf_dire_final, tfidf_radiant_final, scores_all_vect, model_all_vect = train_cv_vectorize(train_full, dire_train_texts,
                                                    radiant_train_texts, feature_cols, params = {'max_iter': 10000})

In [ ]:
test_full = advances_all_test_heroes.merge(chat[chat_cols], 
                                           on='match_id', how='left')
test_full['dire_chat_tokenized'] = prepare_texts(test_full['dire_chat_tokenized'])
test_full['radiant_chat_tokenized'] = prepare_texts(test_full['radiant_chat_tokenized'])



In [ ]:
X_tab_test = csr_matrix(test_full[feature_cols].astype(float).to_numpy())
dire_test_texts = test_full['dire_chat_tokenized'].tolist()
radiant_test_texts = test_full['radiant_chat_tokenized'].tolist()
X_dire_test = tfidf_dire_final.transform(dire_test_texts)
X_radiant_test = tfidf_radiant_final.transform(
    radiant_test_texts)

X_test_all = hstack(
    [X_tab_test, X_dire_test, X_radiant_test], format='csr')

test_pred = model_all_vect.predict_proba(X_test_all)[:, 1]

submission = pd.DataFrame({
    'ID': test_full['match_id'].astype(int),
    'Value': test_pred})
submission.to_csv('submission_all_vect.csv', index = False)

## Pipeline
### The following code contains all the classes and functions required for operation, presented in either their original or modified form. This arrangement is designed to facilitate easy execution.
### Changes:


1. Duration: Instead of being discarded, the 'duration' feature is now treated as a numerical feature. For the test set, missing values ​​are imputed with either zero or the median (both approaches yield identical results, which may suggest that this feature is not of critical importance). Additionally, a flag indicating the absence of a duration value has been added; while essentially a cosmetic detail, this flag has been populated for the entire test set.
2. New Patch: Logic has been implemented to allow for the use of data exclusively from the *new* patch, or alternatively, the use of the *entire* dataset. An additional flag has been introduced to indicate whether a match was played before or after the patch update.
3. Hero Information: It is now possible to incorporate additional hero attributes, such as hero type, primary attribute, and base HP. In practice, however, I have found that including these features tends to degrade cross-validation performance.
4. Binned Trends: Previously, I did not utilize trend metrics based on data bins; this capability has now been implemented. This feature enables the model to learn how the advantage shifts over specific time intervals (bins). In fact, this feature yielded a modest yet positive improvement of 0.0002 during cross-validation, proving to be a useful addition.
5. WinrateEmbedder: I have implemented a cumulative aggregation mechanism for hero win rates and pick rates. To ensure this feature functions correctly, the training dataset (*train*) must be sorted chronologically to preserve the integrity of the cumulative aggregation structure and its subsequent application to the test set. For the test set, I utilize a sorted-array search algorithm (made possible by the prior sorting of the training data) to locate the most recent known timestamp, from which I retrieve the corresponding hero win rates and pick rates. This feature is highly valuable, providing a significant boost during cross-validation; however, its performance on the final submission is not quite as strong.
6. The entire pipeline now runs on `cuml`, although the computation process itself is not particularly fast.

In [4]:
path_files = kagglehub.competition_download('dota-2-hse-ml-1-course-competition-2026')
train = pd.read_csv(path_files + '/matches_df_train.csv')
test = pd.read_csv(path_files + '/matches_df_test.csv')
players = pd.read_csv(path_files + '/player_df.csv')
heroes = pd.read_csv(path_files + '/Constants.Heroes.csv')
advances = pd.read_csv(path_files + '/dota_adv.csv')
chat = pd.read_csv('/kaggle/input/datasets/qwerte123/computed-data-dota-hse/chat_tokenized.csv')

In [5]:
class MissingIndicator(BaseEstimator, TransformerMixin):
  def __init__(self):
    self.columns_names = None

  def fit(self, X, y=None):
    self.columns_names = X.columns.tolist()
    return self

  def transform(self, X):
    X_copy = X.copy()
    for col in self.columns_names:
      X_copy[f'{col}_missing'] = X_copy[col].isna()
    return X_copy

  def get_feature_names_out(self, input_features=None):
    if input_features is None:
      input_features = self.columns_names
    return input_features + ([f'{col}_missing' for col in input_features])


class DateAdder(BaseEstimator, TransformerMixin):
  def __init__(self, date_column='date'):
    self.columns_names = None
    self.date_column = date_column

  def fit(self, X, y=None):
    return self

  def transform(self, X):
    X_copy = X.copy()
    X_copy[self.date_column] = pd.to_datetime(X_copy[self.date_column])
    X_copy['day'] = X_copy[self.date_column].dt.day
    X_copy['day_week'] = X_copy[self.date_column].dt.dayofweek
    X_copy['is_weekend'] = X_copy[self.date_column].dt.weekday >=5
    seasons = {'spring': [3,4,5], 'summer': [6,7,8], 'fall':[9,10,11], 'winter':[12,1,2]}
    reversed_seasons = {l: k  for k, v in seasons.items() for l in v}
    X_copy['season'] = X_copy[self.date_column].dt.month.map(reversed_seasons)
    X_copy = X_copy.drop(self.date_column, axis=1)
    self.columns_names = X_copy.columns.tolist()
    return X_copy

  def get_feature_names_out(self, input_features=None):
    return self.columns_names


In [6]:
def more_agg(df, stat='gold'):
    df = df.copy()

    col_stat = [
        col for col in df.columns
        if stat in col and not any(x in col for x in ['avg', 'std', 'lst', 'kurtosis'])
    ]

    for col in col_stat:
        df[f'{col}_min'] = df[col].apply(lambda x: np.min(x) if len(x) > 0 else np.nan)
        df[f'{col}_max'] = df[col].apply(lambda x: np.max(x) if len(x) > 0 else np.nan)
        df[f'{col}_med'] = df[col].apply(lambda x: np.median(x) if len(x) > 0 else np.nan)
        df[f'{col}_p25'] = df[col].apply(lambda x: np.percentile(x, 0.25) if len(x) > 0 else np.nan)
        df[f'{col}_p75'] = df[col].apply(lambda x: np.percentile(x, 0.75) if len(x) > 0 else np.nan)

    return df

In [7]:
from typing import Iterable
import numpy as np
from collections import defaultdict
from sklearn.preprocessing import MultiLabelBinarizer
from typing import Iterable, Optional
from ast import literal_eval
from scipy.sparse import hstack, csr_matrix

def prepare_texts(series):
    result = []
    for x in series:
        if isinstance(x, list):
            result.append(" ".join(map(str, x)))
        elif x is None or (isinstance(x, float) and np.isnan(x)):
            result.append("")
        else:
            result.append(str(x))
    return result
def base_agg(df, stat='gold'):
    df = df.copy()
    col_stat = [col for col in df.columns if stat in col]
    for col in col_stat:
        df[f'{col}_avg'] = df[col].apply(lambda x: np.mean(x) if len(x) > 0 else np.nan)
        df[f'{col}_std'] = df[col].apply(lambda x: np.std(x) if len(x) > 1 else np.nan)
        df[f'{col}_lst'] = df[col].apply(lambda x: x[-1] if len(x) > 0 else np.nan)
        df[f'{col}_kurtosis'] = df[col].apply(lambda x: pd.Series(x).kurt() if len(x) > 3 else np.nan)

    return df

def print_score(scores):
  for i, s in enumerate(scores):
    print(f'Fold {i+1}: {s}')
  print(f'Average value: {np.mean(scores)}')

def more_agg(df, stat='gold'):
    df = df.copy()

    col_stat = [
        col for col in df.columns
        if stat in col and not any(x in col for x in ['avg', 'std', 'lst', 'kurtosis'])
    ]

    for col in col_stat:
        df[f'{col}_min'] = df[col].apply(lambda x: np.min(x) if len(x) > 0 else np.nan)
        df[f'{col}_max'] = df[col].apply(lambda x: np.max(x) if len(x) > 0 else np.nan)
        df[f'{col}_med'] = df[col].apply(lambda x: np.median(x) if len(x) > 0 else np.nan)
        df[f'{col}_p25'] = df[col].apply(lambda x: np.percentile(x, 0.25) if len(x) > 0 else np.nan)
        df[f'{col}_p75'] = df[col].apply(lambda x: np.percentile(x, 0.75) if len(x) > 0 else np.nan)

    return df
    
def players_merge(df, players, heroes, df_cols, mode='train', add_roles = False,
                 add_primary_attr=False, add_attack_type = False):
    players = players.copy()

    players_grouped = players.groupby(['match_id', 'hero_id'])['hero_id'].count()
    wrong_games = players_grouped[players_grouped != 1].index.get_level_values(0).unique()
    players = players[~players['match_id'].isin(wrong_games)]

    matches_hero0 = players[players['hero_id'] == 0]['match_id'].unique()
    players = players[~players['match_id'].isin(matches_hero0)]

    radiant, dire = range(5), range(128, 133)
    players['team'] = ''
    players.loc[players['player_slot'].isin(radiant), 'team'] = 'radiant'
    players.loc[players['player_slot'].isin(dire), 'team'] = 'dire'

    grouped_matches = players.groupby(['match_id', 'account_id'])['team'].nunique()
    double_players = grouped_matches[
        (grouped_matches > 1) &
        (~grouped_matches.index.get_level_values('account_id').isin([-1, 4294967295]))
    ].reset_index()

    players = players[~players['match_id'].isin(double_players['match_id'].unique())]

    matches = df['match_id'].unique()
    players = players[players['match_id'].isin(matches)]

    hero_encoder = HeroesEncoder(
        add_roles=add_roles,
        add_primary_attr=add_primary_attr,
        add_attack_type=add_attack_type
    )
    hero_encoder.fit(heroes)

    match_heropool_df = hero_encoder.transform(players)
    match_heropool_df['match_id'] = match_heropool_df['match_id'].astype(df['match_id'].dtype)

    if mode == 'train':
        heropool_features = pd.merge(df[df_cols], match_heropool_df, how='inner', on='match_id')
    else:
        heropool_features = pd.merge(df[df_cols], match_heropool_df, how='left', on='match_id')
        heropool_features.fillna(0, inplace=True)

    return heropool_features
    


class HeroesEncoder:
    def __init__(
        self,
        hero_ids=None,
        add_roles=False,
        add_primary_attr=False,
        add_attack_type=False):
        
        self.hero_ids = hero_ids
        self.add_roles = add_roles
        self.add_primary_attr = add_primary_attr
        self.add_attack_type = add_attack_type

    def _parse_roles(self, x):
        if isinstance(x, list):
            return [str(i).strip() for i in x]
        if pd.isna(x):
            return []
        if isinstance(x, str):
            x = x.strip()
            if x == '':
                return []
            try:
                val = literal_eval(x)
                if isinstance(val, list):
                    return [str(i).strip() for i in val]
            except Exception:
                return []
        return []

    def fit(self, heroes_df, y=None):
        heroes_df = heroes_df.copy()
        heroes_df = heroes_df.drop_duplicates(subset=['id'])

        if self.hero_ids is not None:
            self.unique_heroes_ = list(self.hero_ids)
        else:
            self.unique_heroes_ = sorted(
                heroes_df['id'].dropna().astype(int).unique().tolist()
            )

        self.heroes_to_id_ = {
            hero_id: i for i, hero_id in enumerate(self.unique_heroes_)
        }

        heroes_df = heroes_df[heroes_df['id'].isin(self.unique_heroes_)].copy()
        heroes_df['id'] = heroes_df['id'].astype(int)
        heroes_df = heroes_df.set_index('id')

        if self.add_roles:
            roles = heroes_df['roles'].apply(self._parse_roles)
            self.mlb_ = MultiLabelBinarizer()
            roles_arr = self.mlb_.fit_transform(roles)
            self.roles_encoded_ = pd.DataFrame(
                roles_arr,
                index=heroes_df.index,
                columns=[f'role_{c}' for c in self.mlb_.classes_],
                dtype=np.int8
            )
        else:
            self.roles_encoded_ = None

        if self.add_primary_attr:
            self.attr_encoded_ = pd.get_dummies(
                heroes_df['primary_attr'].fillna('unknown'),
                prefix='attr',
                dtype=np.int8
            )
            self.attr_encoded_.index = heroes_df.index
        else:
            self.attr_encoded_ = None

        if self.add_attack_type:
            self.attack_encoded_ = pd.get_dummies(
                heroes_df['attack_type'].fillna('unknown'),
                prefix='attack',
                dtype=np.int8
            )
            self.attack_encoded_.index = heroes_df.index
        else:
            self.attack_encoded_ = None

        return self

    def transform(self, X, y=None):
        required_cols = {'match_id', 'hero_id', 'player_slot'}
        missing = required_cols - set(X.columns)
        if missing:
            raise ValueError(f'Missing columns: {sorted(missing)}')

        X = X.copy()

        slots = X['player_slot'].to_numpy()
        radiant_mask = (slots >= 0) & (slots <= 4)
        dire_mask = (slots >= 128) & (slots <= 132)
        valid_mask = radiant_mask | dire_mask

        X = X.loc[valid_mask].copy()
        X['hero_id'] = pd.to_numeric(X['hero_id'], errors='coerce')
        X = X[X['hero_id'].isin(self.heroes_to_id_)].copy()
        X['hero_id'] = X['hero_id'].astype(int)

        if X.empty:
            return pd.DataFrame(columns=self.get_feature_names_out())

        unique_matches = X['match_id'].unique()
        matches_to_id = {match_id: i for i, match_id in enumerate(unique_matches)}

        rows = X['match_id'].map(matches_to_id).to_numpy()
        cols = X['hero_id'].map(self.heroes_to_id_).to_numpy()

        slots = X['player_slot'].to_numpy()
        signs = np.where((slots >= 0) & (slots <= 4), 1, -1)

        n_matches = len(unique_matches)
        n_heroes = len(self.unique_heroes_)

        hero_data = np.zeros((n_matches, n_heroes), dtype=np.int8)
        hero_data[rows, cols] = signs

        result = pd.DataFrame({'match_id': unique_matches})

        if self.add_roles:
            role_data = np.zeros((n_matches, self.roles_encoded_.shape[1]), dtype=np.int16)
            role_lookup = self.roles_encoded_.loc[X['hero_id']].to_numpy(dtype=np.int8)
            np.add.at(role_data, rows, role_lookup * signs[:, None])

            role_df = pd.DataFrame(role_data, columns=self.roles_encoded_.columns)
            result = pd.concat([result, role_df], axis=1)

        if self.add_primary_attr:
            attr_data = np.zeros((n_matches, self.attr_encoded_.shape[1]), dtype=np.int16)
            attr_lookup = self.attr_encoded_.loc[X['hero_id']].to_numpy(dtype=np.int8)
            np.add.at(attr_data, rows, attr_lookup * signs[:, None])

            attr_df = pd.DataFrame(attr_data, columns=self.attr_encoded_.columns)
            result = pd.concat([result, attr_df], axis=1)

        if self.add_attack_type:
            attack_data = np.zeros((n_matches, self.attack_encoded_.shape[1]), dtype=np.int16)
            attack_lookup = self.attack_encoded_.loc[X['hero_id']].to_numpy(dtype=np.int8)
            np.add.at(attack_data, rows, attack_lookup * signs[:, None])

            attack_df = pd.DataFrame(attack_data, columns=self.attack_encoded_.columns)
            result = pd.concat([result, attack_df], axis=1)

        hero_df = pd.DataFrame(
            hero_data,
            columns=[f'hero_{hero_id}' for hero_id in self.unique_heroes_]
        )

        result = pd.concat([result, hero_df], axis=1)
        return result

    def get_feature_names_out(self):
        cols = ['match_id']

        if self.add_roles:
            cols.extend(self.roles_encoded_.columns.tolist())

        if self.add_primary_attr:
            cols.extend(self.attr_encoded_.columns.tolist())

        if self.add_attack_type:
            cols.extend(self.attack_encoded_.columns.tolist())

        cols.extend([f'hero_{hero_id}' for hero_id in self.unique_heroes_])
        return cols


class TrendTransformer:

    def __init__(self, columns: Iterable[str], method = 'delta', join_col = 'match_id'):
        self.columns = columns
        self.method = method
        self.join_col = join_col
        self.slope = None
        self.interception = None
        self.r2 = None

    def fit(self, X, y=None):
      return self

    def transform(self, values):
        self.r2 = {col: [] for col in self.columns}
        self.slope = {col: [] for col in self.columns}
        self.interception = {col: [] for col in self.columns}
        for col in self.columns:
            for cur_values in values[col]:
                cur_val = np.asarray(cur_values)
                X = np.arange(len(cur_values))
                if len(cur_val) == 0:
                  cur_slope = np.nan
                  cur_interception = np.nan
                  cur_r2 = np.nan

                if len(cur_val) == 1:
                  cur_slope = np.nan
                  cur_interception = cur_val[0]
                  cur_r2 = np.nan

                if self.method == 'delta':
                  if len(cur_val) > 1:
                    cur_slope = (cur_val[-1] - cur_val[0])/(len(X) - 1) # y = kx + b
                    cur_interception = cur_val[0]
                    y_mean = np.mean(cur_val)
                    y_pred = cur_slope * X + cur_interception
                    RSS = np.sum((y_pred - cur_val)**2)
                    TSS = np.sum((cur_val - y_mean)**2)
                    cur_r2 = 1 - RSS/TSS if TSS != 0 else np.nan

                elif self.method == 'OLS':
                  if len(cur_val) > 1:
                    X_ols = np.column_stack([X, np.full(fill_value = 1, shape=(cur_val.shape))])
                    X_ols_t = X_ols.T
                    beta = np.linalg.inv(X_ols_t @ X_ols) @ X_ols_t @ cur_val
                    cur_slope = beta[0]
                    cur_interception = beta[1]
                    y_mean = np.mean(cur_val)
                    y_pred = cur_slope * X + cur_interception
                    RSS = np.sum((y_pred - cur_val)**2)
                    TSS = np.sum((cur_val - y_mean)**2)
                    cur_r2 = 1 - RSS/TSS if TSS != 0 else np.nan
                self.r2[col].append(cur_r2)
                self.slope[col].append(cur_slope)
                self.interception[col].append(cur_interception)

        res = pd.DataFrame(values[self.join_col])
        for col in self.columns:
          res[f'{col}_r2'] = self.r2[col]
          res[f'{col}_slope'] = self.slope[col]
          res[f'{col}_interception'] = self.interception[col]
        return res

class BinTransformer:
  def __init__(self, columns: Iterable[str], join_col = 'match_id', number_bins=4):
    self.columns = columns
    self.join_col = join_col
    self.number_bins = number_bins

  def fit(self, X, y=None):
    return self

  def transform(self, values):
    res = defaultdict(list)
    for col in self.columns:
        for cur_values in values[col]:
          cur_values = np.asarray(cur_values)
          if len(cur_values) == 0:
              for i in range(self.number_bins):
                res[f'{col}_bin_{i}'].append([np.nan])
              continue
          step = len(cur_values)//self.number_bins
          for i in range(self.number_bins):
                bin_i = cur_values[i*step:(i+1)*step]
                res[f'{col}_bin_{i}'].append(bin_i)
    res = pd.DataFrame(res)
    res[self.join_col] = values[self.join_col]
    return res

In [8]:
import numpy as np
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression as skLogReg
from sklearn.model_selection import TimeSeriesSplit
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack, csr_matrix
from sklearn.preprocessing import StandardScaler

class DotaPipeline:
    def __init__(
        self, 
        train, 
        test,
        players = None,
        heroes = None, 
        chat = None, 
        advances = None,
        model = skLogReg,
        model_params = {},
        new_patch_only = False,
        use_base_prep = True,
        use_durationflag = True,
        use_heroes = True,
        use_chat = True,
        use_advances = True,
        use_advances_trends = True,
        use_advances_more_agg = True,
        use_advances_bins = True,
        use_advances_bins_trends = True,
        method_trends = 'OLS',
        duration_method = 'median',
        text_min_df = 1,
        text_max_features = 3000        
    ):
        
        self.train = train
        self.test = test
        self.chat = chat
        self.heroes = heroes
        self.players = players
        self.advances = advances
        self.new_patch_only = new_patch_only
        self.use_base_prep = use_base_prep
        self.use_durationflag = use_durationflag
        self.use_heroes = use_heroes
        self.use_chat = use_chat
        self.use_advances = use_advances
        self.use_advances_trends = use_advances_trends
        self.use_advances_more_agg = use_advances_more_agg
        self.use_advances_bins = use_advances_bins
        self.use_advances_bins_trends = use_advances_bins_trends
        self.model = model
        self.model_params = model_params
        self.text_min_df = text_min_df
        self.text_max_features=text_max_features
        self.method_trends = method_trends
        self.duration_method = duration_method
        self.preprocessor_ = None
        self.final_model = None
        self.advances_all = None
    
    def base_prep(self):
        passthrough_columns = ['match_id']
        if not self.new_patch_only:
            passthrough_columns.append('is_new_patch')
    
        if self.use_durationflag:
            passthrough_columns.append('duration_missing')
    
        num_features = ['avg_mmr', 'duration']
    
        region_features = ['region']
        game_mode_features = ['game_mode']
        date_features = ['date']
    
        imputer = SimpleImputer(strategy='median')
        sqrt_transformer = FunctionTransformer(
            func=np.sqrt,
            feature_names_out='one-to-one'
        )
    
        pipe_num = Pipeline([
            ('missing_ind', MissingIndicator()),
            ('imputer', imputer),
            ('transformer', sqrt_transformer),
            ('scaler', RobustScaler())
        ])
    
        ohe = OneHotEncoder(
            sparse_output=False,
            handle_unknown='ignore'
        )
    
        pipe_reg = Pipeline([('ecoder', ohe)])
        pipe_game_mode = Pipeline([('encoder', ohe)])
    
        pipe_date = Pipeline([
            ('adder', DateAdder()),
            ('ecoder', ColumnTransformer([
                ('weekend', 'passthrough', ['is_weekend']),
                ('ohe', ohe, ['day', 'day_week', 'season'])
            ], verbose_feature_names_out=False))
        ])
    
        preprocessor = ColumnTransformer([
            ('passthrough', 'passthrough', passthrough_columns),
            ('num', pipe_num, num_features),
            ('reg', pipe_reg, region_features),
            ('game_mode', pipe_game_mode, game_mode_features),
            ('date', pipe_date, date_features)
        ], verbose_feature_names_out=False)
    
        return preprocessor
            
    def chat_merge(self, df):
        chat = self.chat.copy()
        chat = chat.fillna('')
        res = df.merge(chat, how='left', on = 'match_id')
        return res
        
    def players_merge(self, df, players, heroes, df_cols, mode='train'):
        return players_merge(df, players, heroes, df_cols, mode = mode)

    def advances_prep(self):
        if not(self.advances_all is None):
            return self.advances_all
        
        advances = self.advances.copy()
        advances['radiant_gold_adv'] = advances['radiant_gold_adv'].apply(
            lambda x: x.replace('[', '').replace(']', '').split())
        
        advances['radiant_gold_adv'] = advances['radiant_gold_adv'].apply(
            lambda x: list(map(int, x)))
        
        advances['radiant_exp_adv'] = advances['radiant_exp_adv'].apply(
            lambda x: x.replace('[', '').replace(']', '').split())
        
        advances['radiant_exp_adv'] = advances['radiant_exp_adv'].apply(
            lambda x: list(map(int, x)))
        
        advances_nonan = advances.copy()

        advances_based = base_agg(advances_nonan)
        advances_based = base_agg(advances_based, 'exp')

        advances_based['dire_exp_adv_lst'] = advances_based['radiant_exp_adv_lst'].apply(
            lambda x: -x if x < 0 else 0)
        advances_based['dire_gold_adv_lst'] = advances_based['radiant_gold_adv_lst'].apply(
            lambda x: -x if x < 0 else 0)

        advances_based['radiant_exp_adv_lst'] = advances_based['radiant_exp_adv_lst'].apply(
            lambda x: x if x > 0 else 0)
        advances_based['radiant_gold_adv_lst'] = advances_based['radiant_gold_adv_lst'].apply(
            lambda x: x if x > 0 else 0)
        
        advances_based['is_nan'] = advances_based['radiant_exp_adv_kurtosis'].isnull()
        
        scale_cols_based = advances_based.columns.drop(['match_id', 'is_nan', 
                                                        'radiant_gold_adv', 
                                                        'radiant_exp_adv'])
        scaler = StandardScaler()
        scaled = scaler.fit_transform(advances_based[scale_cols_based])
        scaled = pd.DataFrame(scaled, columns = scaler.get_feature_names_out())
        advances_based[scale_cols_based] = scaled

        if self.use_advances_more_agg:
            advances_based_more = more_agg(advances_based, 'gold')
            advances_based_more = more_agg(advances_based_more, 'exp')
        else:
            advances_based_more = advances_based
            
        scale_cols_based_more = advances_based_more.columns.drop(['match_id', 'is_nan', 'radiant_gold_adv', 'radiant_exp_adv'])
        scaled = scaler.fit_transform(advances_based_more[scale_cols_based_more])
        scaled = pd.DataFrame(scaled, columns = scaler.get_feature_names_out())
        advances_based_more[scale_cols_based_more] = scaled
        if self.use_advances_trends:
            Tt = TrendTransformer(['radiant_gold_adv', 'radiant_exp_adv'], method= self.method_trends)
            trends = Tt.transform(advances_based_more)
    
            to_be_scaled = trends.columns.drop('match_id')
            trends_scaled = scaler.fit_transform(trends[to_be_scaled])
            trends_scaled = pd.DataFrame(trends_scaled, columns=scaler.get_feature_names_out())
            trends[to_be_scaled] = trends_scaled
            
            advances_trends = pd.merge(advances_based_more, trends, how='left', 
                                       on='match_id')
        else:
            advances_trends = advances_based_more
            
        if self.use_advances_bins:
            Bt = BinTransformer(['radiant_gold_adv', 'radiant_exp_adv'], number_bins=4)
            bins_cols = Bt.transform(advances_trends)
    
            bins_based = base_agg(bins_cols)
            bins_based = base_agg(bins_based,'exp')
            
            if self.use_advances_more_agg:
                bins_based = more_agg(bins_based)
                bins_based = more_agg(bins_based,'exp')
                
            if self.use_advances_bins_trends:
                cols_trend = [col for col in bins_based.columns if 'bin' in col[-6:]]
                Tt_bins = TrendTransformer(cols_trend, method= self.method_trends)
                trends_bins = Tt_bins.transform(bins_based)
                bins_based = bins_based.merge(trends_bins, how = 'left', on='match_id')
            
            scaling_cols = bins_based.columns.drop([col for col in bins_based.columns if 'bin' in col[-6:] or col == 'match_id'])
            bins_based_scaled = scaler.fit_transform(bins_based[scaling_cols])
            bins_based[scaling_cols] = bins_based_scaled
            advances_bins = advances_based_more.merge(bins_based, how='left', on='match_id')
        else:
            advances_bins = advances_based_more
        diff_cols = advances_trends.columns.difference(advances_bins.columns).tolist() \
            + ['match_id']
        advances_all = advances_bins.merge(advances_trends[diff_cols], on = 'match_id',
                                           how = 'left')

        return advances_all.fillna(0)

    def add_features_train(self, train):
        train_cp = train.copy()
        train_cp['is_new_patch'] = train_cp['date'] >= '2024-05-23'
    
        if self.use_durationflag:
            train_cp['duration_missing'] = train_cp['duration'].isna().astype(int)
    
        if self.duration_method == 'median':
            self.duration_fill_value_ = train_cp['duration'].median()
        elif self.duration_method == 'zero':
            self.duration_fill_value_ = 0
        
        train_cp['duration'] = train_cp['duration'].fillna(self.duration_fill_value_)
    
        return train_cp

    def add_features_test(self, test):
        test_cp = test.copy()
        test_cp['is_new_patch'] = test_cp['date'] >= '2024-05-23'
    
        if self.use_durationflag:
            test_cp['duration_missing'] = 1
    
        if self.duration_method == 'median':
            fill_value = self.duration_fill_value_
        elif self.duration_method == 'zero':
            fill_value = 0
       
        test_cp['duration'] = fill_value
        return test_cp

    def make_base_df(self, df, fit=False):
        if self.use_base_prep:
            if fit:
                self.preprocessor_ = self.base_prep()
                X = self.preprocessor_.fit_transform(df)
            else:
                X = self.preprocessor_.transform(df)
    
            cols = (
                self.preprocessor_['passthrough'].get_feature_names_out().tolist()
                + self.preprocessor_['num'].get_feature_names_out().tolist()
                + self.preprocessor_['reg'].get_feature_names_out().tolist()
                + self.preprocessor_['game_mode'].get_feature_names_out().tolist()
                + self.preprocessor_['date'].get_feature_names_out().tolist()
            )
    
            X_df = pd.DataFrame(X, columns=cols, index=df.index)
        else:
            X_df = pd.DataFrame({'match_id': df['match_id'].values}, index=df.index)
    
        if 'radiant_win' in df.columns:
            X_df['radiant_win'] = df['radiant_win'].values
    
        if self.use_heroes:
            mode = 'train' if fit else 'test'
            X_df = self.players_merge(X_df, self.players, self.heroes, X_df.columns, mode)
    
        if self.use_chat:
            X_df = self.chat_merge(X_df)
    
        if self.use_advances:
            self.advances_all = self.advances_prep()
            X_df = X_df.merge(self.advances_all, how='left', on='match_id')
    
        return X_df
    
    def train_cv(self):
        if self.use_base_prep:
            if self.new_patch_only:
                train_patch = self.train[self.train['date'] >= '2024-05-23'].copy()
                train_patch = self.add_features_train(train_patch)
            else:
                train_patch = self.add_features_train(self.train)
        else:
            train_patch = self.train.copy()
        
        cv = TimeSeriesSplit(n_splits=5)
        scores = []
        train_full = self.make_base_df(train_patch, fit=True)   
        drop_cols = ['match_id', 'radiant_win']
        if self.use_chat:
            drop_cols.extend(self.chat.columns)

        if self.use_advances:
            drop_cols.extend(['radiant_gold_adv', 'radiant_exp_adv'])
            drop_cols.extend([i for i in train_full.columns if 'bin' in i[-6:]])
        feature_cols = [c for c in train_full.columns if c not in drop_cols]
    
        X_train_df = train_full[feature_cols].astype(float)
        y_train_df = train_full['radiant_win']
        for fold, (train_index, val_index) in enumerate(cv.split(train_full)):
            X_train, X_val = X_train_df.iloc[train_index], X_train_df.iloc[val_index]
            y_train, y_val = y_train_df.iloc[train_index], y_train_df.iloc[val_index]
            parts_train = [csr_matrix(X_train.to_numpy())]
            parts_val = [csr_matrix(X_val.to_numpy())]
            if self.use_chat:
                dire_train = train_full['dire_chat_tokenized'].iloc[train_index]
                rad_train = train_full['radiant_chat_tokenized'].iloc[train_index]
                dire_val = train_full['dire_chat_tokenized'].iloc[val_index]
                rad_val = train_full['radiant_chat_tokenized'].iloc[val_index]
                tfidf_dire = TfidfVectorizer(
                    min_df=self.text_min_df,
                    max_features=self.text_max_features
                )
                tfidf_radiant = TfidfVectorizer(
                    min_df=self.text_min_df,
                    max_features=self.text_max_features
                )

                X_train_dire = tfidf_dire.fit_transform(dire_train)
                X_val_dire = tfidf_dire.transform(dire_val)

                X_train_radiant = tfidf_radiant.fit_transform(rad_train)
                X_val_radiant = tfidf_radiant.transform(rad_val)
                parts_train.extend([X_train_dire, X_train_radiant])
                parts_val.extend([X_val_dire, X_val_radiant])


            X_train = hstack(parts_train, format="csr")
            X_val = hstack(parts_val, format="csr")
            
            cur_model = self.model(**self.model_params)
            cur_model.fit(X_train, y_train)
            predict_val = cur_model.decision_function(X_val)
            score = gini(y_val, predict_val)
            scores.append(score)
        print_score(scores)
     
        return self

    def fit(self):       
        if self.new_patch_only:
            train_patch = self.train[self.train['date'] >= '2024-05-23']
        else:
            train_patch = self.add_features_train(self.train)
    
        train_full = self.make_base_df(train_patch, fit=True)
        drop_cols = ['match_id', 'radiant_win']
        if self.use_chat:
            drop_cols.extend(self.chat.columns)
            
        if self.use_advances:
            drop_cols.extend(['radiant_gold_adv', 'radiant_exp_adv'])
            drop_cols.extend([i for i in train_full.columns if 'bin' in i[-6:]])
            
        feature_cols = [c for c in train_full.columns if c not in drop_cols]
    
        X_train_df = train_full[feature_cols].astype(float)
        y_train_df = train_full['radiant_win']
        parts_train = [csr_matrix(X_train_df.values)]
    
        if self.use_chat:
    
            dire_chat = train_full['dire_chat_tokenized']
            radiant_chat = train_full['radiant_chat_tokenized']
            
            self.tfidf_dire = TfidfVectorizer(
                min_df=self.text_min_df,
                max_features=self.text_max_features
            )
    
            self.tfidf_radiant = TfidfVectorizer(
                min_df=self.text_min_df,
                max_features=self.text_max_features
            )
    
            X_dire = self.tfidf_dire.fit_transform(dire_chat)
            X_radiant = self.tfidf_radiant.fit_transform(radiant_chat)
    
            parts_train.extend([X_dire, X_radiant])
    
        X_train = hstack(parts_train, format="csr")
    
        final_model = self.model(**self.model_params)
        final_model.fit(X_train, y_train_df)
    
        self.final_model = final_model
    
        return self

    def make_pred(self):
        test = self.test.copy()
        if self.use_base_prep:
            test = self.add_features_test(test)
    
        test_full = self.make_base_df(test, fit=False)
        ids = test_full['match_id']
        drop_cols = ['match_id']
    
        if self.use_advances:
            drop_cols.extend(['radiant_gold_adv', 'radiant_exp_adv'])
            drop_cols.extend([i for i in test_full.columns if 'bin' in i[-6:]])
    
        if self.use_chat:
            drop_cols.extend(self.chat.columns)
    
        feature_cols = [c for c in test_full.columns if c not in drop_cols]
        X_test_df = test_full[feature_cols].astype(float)
    
        parts_test = [csr_matrix(X_test_df.values)]
    
        if self.use_chat:
            dire_chat = prepare_texts(test_full['dire_chat_tokenized'])
            radiant_chat = prepare_texts(test_full['radiant_chat_tokenized'])
    
            X_dire = self.tfidf_dire.transform(dire_chat)
            X_radiant = self.tfidf_radiant.transform(radiant_chat)
    
            parts_test.extend([X_dire, X_radiant])
    
        X_test = hstack(parts_test, format="csr")
        pred = self.final_model.predict_proba(X_test)[:, 1]
    
        res_df = pd.DataFrame({
            'ID': ids.astype(int),
            'Value': pred})
    
        return res_df

In [11]:
pipeline = DotaPipeline(
    train=train.sort_values('date'),
    test=test,
    players=players,
    heroes=heroes,
    chat=chat,
    advances=advances,

    model=cmLogReg,
    model_params={
        'max_iter': 20000,
    },

    new_patch_only=False,
    use_base_prep=True,
    use_durationflag=True,
    use_heroes=True,
    use_chat=True,
    use_advances=True,
    use_advances_trends=True,
    use_advances_more_agg=True,
    use_advances_bins=True,
    use_advances_bins_trends=True,
    method_trends='OLS',
    duration_method='median',
    text_min_df=1,
    text_max_features=30000
)
pipeline.train_cv()
pipeline.fit()
sub = pipeline.make_pred()
sub.to_csv('sub_pipe_last.csv',index=False)

Fold 1: 0.41113385707257066
Fold 2: 0.42048952748424484
Fold 3: 0.42452600384345374
Fold 4: 0.42575853463851754
Fold 5: 0.4279328795495432
Average value: 0.42196816051766606
